In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
current_dir = Path().resolve()
sys.path.append(current_dir.parent.parent.as_posix())

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm

import utils
from data_io import DataIO
from psth_utils import *
from psth_params import *

data_dir   = Path(r'/media/aleong/Audrey-experiments1') / 'Experiments' / 'dataset'
figure_dir = data_dir / 'Figures_summary'
os.makedirs(figure_dir, exist_ok=True)
print('Setup done.')


## Session configuration

In [ ]:
# ── Per-session configuration ─────────────────────────────────────────────────
# recording_nrs : [noblocker, acet, acet+lap4, washout] recording indices
# chirp_rec_nb  : 0-based index of the chirp recording used for cell typing
SESSION_CONFIG = {
    '260415_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260423_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260424_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260519_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
    '260520_A': {'recording_nrs': [5, 7, 9, 11], 'chirp_rec_nb': 2},
}

BLOCKER_ORDER = ['noblocker', 'acet', 'acet lap4', 'washout']
BLOCKER_LABEL = {
    'noblocker': 'No blocker',
    'acet':      'ACET',
    'acet lap4': 'ACET + LAP4',
    'washout':   'Washout',
}

CT_ORDER  = ['ON', 'OFF', 'ON-OFF', 'Unknown']
CT_COLORS = {'ON': '#E63946', 'OFF': '#457B9D', 'ON-OFF': '#9B59B6', 'Unknown': '#95A5A6'}

RESP_COLOR      = {'excitatory': '#E63946', 'inhibitory': '#457B9D'}
RESP_COLORS_CNT = {'excitatory': '#E63946', 'inhibitory': '#457B9D'}

SESSION_COLOR_LIST = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
SESSION_COLORS_MAP = dict(zip(SESSION_CONFIG.keys(), SESSION_COLOR_LIST))


## Load all sessions and compute PSTHs
For each session: load data, find preferred electrode per cell, compute PSTHs, accumulate into `df` (significant responses) and `zscore_store_all` (all conditions).  
Cell typing is also loaded per session when available.


In [ ]:
df_rows          = []
zscore_store_all = {}   # key: (cluster_id, blocker, prr, power)
typing_dfs       = {}   # key: session_id

for session_id, cfg in SESSION_CONFIG.items():
    print(f'\n{"="*60}')
    print(f'  {session_id}')
    print(f'{"="*60}')

    recording_nrs = cfg['recording_nrs']
    chirp_rec_nb  = cfg['chirp_rec_nb']

    # ── Load session ────────────────────────────────────────────
    data_io  = DataIO(data_dir)
    loadname = data_dir / f'{session_id}_cells.csv'
    data_io.load_session(session_id, load_pickle=True, load_waveforms=False)
    cells_df = pd.read_csv(loadname, header=[0, 1], index_col=0)

    # ── Select blocker-condition recordings ─────────────────────
    selected_rec_names = []
    for r in recording_nrs:
        for rec_id in data_io.recording_ids:
            if f'_{r:03d}_' in rec_id:
                selected_rec_names.append(rec_id)
                break
    if not selected_rec_names:
        print('  No matching recordings found — skipping.')
        continue
    print(f'  Recordings: {selected_rec_names}')

    # ── Preferred electrode per cell ────────────────────────────
    pref_ec_dict = {}
    for cluster_id in data_io.cluster_df.index.values:
        pref_ec, n_sig_pref_ec = None, None
        for ec in data_io.burst_df.electrode.unique():
            if pd.isna(ec):
                continue
            tids = data_io.burst_df.query(f'electrode == {float(ec)}').train_id.unique()
            n_sig = sum(
                1 for tid in tids
                if cells_df.loc[cluster_id, (tid, 'is_significant')] == True
            )
            if n_sig > 1 and (pref_ec is None or n_sig > n_sig_pref_ec):
                pref_ec, n_sig_pref_ec = ec, n_sig
        pref_ec_dict[cluster_id] = pref_ec

    n_responsive = sum(1 for v in pref_ec_dict.values() if v is not None)
    print(f'  Cells with preferred electrode: {n_responsive} / {len(pref_ec_dict)}')

    # ── PSTH computation ─────────────────────────────────────────
    cluster_ids = data_io.cluster_df.index.values

    for cluster_id in tqdm(cluster_ids, desc=f'  PSTHs'):
        ec = pref_ec_dict.get(cluster_id)
        if ec is None:
            continue

        for rec_name in selected_rec_names:
            d_select = data_io.burst_df.query(
                'electrode == @ec and recording_name == @rec_name'
            ).copy()
            if d_select.empty:
                continue

            blocker = d_select.Blocker.iloc[0]

            for prr in sorted(d_select.laser_pulse_repetition_rate.unique()):
                d_prr = d_select.query('laser_pulse_repetition_rate == @prr')

                for power in sorted(d_prr.laser_power.unique()):
                    d_pow = d_prr.query('laser_power == @power')
                    if d_pow.empty:
                        continue

                    tid      = d_pow.iloc[0].train_id
                    row0     = data_io.burst_df.query('train_id == @tid').iloc[0]
                    rec_id   = str(row0.rec_id)
                    stimtype = str(row0.stimtype)
                    bd       = float(row0.laser_burst_duration)

                    if stimtype in ('laser', 'padmd'):
                        burst_onsets = data_io.burst_df.query('train_id == @tid').laser_burst_onset.values
                    elif stimtype == 'dmd':
                        burst_onsets = data_io.burst_df.query('train_id == @tid').dmd_burst_onset.values
                    else:
                        continue

                    spiketrain = data_io.spiketimes[rec_id][cluster_id]

                    binned = []
                    for onset in burst_onsets:
                        idx = np.where(
                            (spiketrain >= onset + t_edges[0]) &
                            (spiketrain <  onset + t_edges[-1])
                        )[0]
                        counts, _ = np.histogram(spiketrain[idx] - onset, bins=t_edges)
                        binned.append(counts)

                    if not binned:
                        continue

                    binned        = np.vstack(binned)
                    spike_counts  = binned.sum(axis=0)
                    rate          = spike_counts / (len(binned) * (BIN_SIZE_MS / 1000))

                    baseline_mask = (t_centers >= -PRE_TIME_MS) & (t_centers < 0)
                    baseline_rate = rate[baseline_mask].mean()
                    if baseline_rate < MIN_BASELINE_HZ:
                        continue

                    smooth_sd   = get_adaptive_smooth_sd(baseline_rate)
                    rate_smooth = gaussian_filter1d(rate, smooth_sd)
                    mu          = rate_smooth[baseline_mask].mean()
                    sd          = rate_smooth[baseline_mask].std(ddof=1)

                    zscore      = (rate_smooth - mu) / sd if sd > 0 else np.zeros_like(rate_smooth)
                    zscore_norm = np.tanh(zscore / 3)

                    spike_counts_bl = spike_counts[baseline_mask]
                    results         = detect_latency(rate_smooth, mu, sd, spike_counts, spike_counts_bl)

                    # Mean post-stim rate in MEAN_WINDOW_MS
                    post_mask       = (t_centers >= MEAN_WINDOW_MS[0]) & (t_centers <= MEAN_WINDOW_MS[1])
                    mean_post_rate  = float(rate[post_mask].mean())

                    cond_key = (cluster_id, blocker, float(prr), float(power))
                    zscore_store_all[cond_key] = {
                        'session'       : session_id,
                        'zscore'        : zscore_norm,
                        'rate'          : rate_smooth,
                        'rate_raw'      : rate,
                        'mu'            : mu,
                        'sd'            : sd,
                        'latency'       : results['latency_ms'],
                        'response_type' : results['resp_type'],
                        'significant'   : results['significant'],
                        'bd'            : bd,
                        'mean_post_rate': mean_post_rate,
                    }

                    if not results['significant']:
                        continue

                    df_rows.append({
                        'session'       : session_id,
                        'cluster_id'    : cluster_id,
                        'blocker'       : blocker,
                        'prr'           : float(prr),
                        'power'         : float(power),
                        'response_type' : results['resp_type'],
                        'latency_ms'    : results['latency_ms'],
                        'baseline_rate' : mu,
                        'mean_post_rate': mean_post_rate,
                        'p_value'       : results['p_value'],
                    })

    # ── Load cell typing ─────────────────────────────────────────
    typing_directory = (
        Path(f'/media/aleong/Audrey-experiments1/Experiments/{session_id}/Analysis')
        / f'CellTyping_Analysis_rec_{chirp_rec_nb}'
    )
    typing_data_path = typing_directory / f'{session_id}_PA_ACET_1_cell_typing_data'
    try:
        typing_df = load_obj_as_df(typing_data_path)
        new_index = [f'uid_{session_id.split("_")[0]}_{i:03}' for i in typing_df.index]
        typing_df.index = new_index
        typing_dfs[session_id] = typing_df
        print(f'  Cell typing loaded: {len(typing_df)} cells')
    except Exception as e:
        print(f'  Cell typing not available: {e}')

df = pd.DataFrame(df_rows)
print(f'\nTotal significant responses across all sessions: {len(df)}')
if not df.empty:
    print(df.groupby(['blocker', 'response_type']).size().unstack(fill_value=0))


## Shared axis values and cell-type mapping

In [ ]:
# ── Collect all PRR / power values present across sessions ────────────────────
prr_vals   = sorted(df['prr'].unique())    if not df.empty else []
power_vals = sorted(df['power'].unique())  if not df.empty else []

df['prr_str']   = df['prr'].apply(lambda x: f'{int(x)} Hz')
df['power_str'] = df['power'].apply(lambda x: f'{int(x)} µW')
prr_order_str   = [f'{int(p)} Hz'  for p in prr_vals]
power_order_str = [f'{int(p)} µW'  for p in power_vals]

# ── Build combined typing lookup: cluster_id → cell_type / full_type_name ────
def _get_full_type(cell_type_value):
    """Return the full Baden type name (e.g. 'OFF alpha')."""
    if cell_type_value is None:
        return None
    if isinstance(cell_type_value, dict):
        return cell_type_value.get('name', None)
    if isinstance(cell_type_value, str):
        try:
            d = eval(cell_type_value)
            if isinstance(d, dict):
                return d.get('name', cell_type_value)
        except Exception:
            pass
        return cell_type_value
    return None

cid_to_ct       = {}
cid_to_fulltype = {}
for session_id, typing_df in typing_dfs.items():
    for cid in typing_df.index:
        btype = typing_df.loc[cid, 'baden_type']
        cid_to_ct[cid]       = extract_cell_type(btype) or 'Unknown'
        cid_to_fulltype[cid] = _get_full_type(btype)    or 'Unknown'

# ── Attach cell types to df ───────────────────────────────────────────────────
df['cell_type']  = df['cluster_id'].map(lambda c: cid_to_ct.get(c, 'Unknown'))
df['full_type']  = df['cluster_id'].map(lambda c: cid_to_fulltype.get(c, 'Unknown'))

HAS_TYPING = bool(typing_dfs)
print(f'Typing available: {HAS_TYPING}')
print(f'PRR values : {prr_vals}')
print(f'Power values: {power_vals}')


In [ ]:
# ── Universal blocker colour palette ─────────────────────────────────────────
BLK_COLOR = {
    'noblocker': '#000000',   # black
    'acet':      '#69529d',   # purple
    'acet lap4': '#f0861f',   # orange
    'washout':   '#27AE60',   # green
}
BLK_LABEL = {
    'noblocker': 'No blocker',
    'acet':      'ACET',
    'acet lap4': 'ACET+LAP4',
    'washout':   'Washout',
}

## Response latency — by PRR and laser power, per blocker condition
One figure per response type (excitatory / inhibitory).  
Rows = blocker conditions; left panel: x = PRR, right panel: x = laser power.  
Each point is one significant response; all sessions pooled.


In [ ]:
if df.empty:
    print('No significant responses — skipping latency plots.')
else:
    for resp_type in ['excitatory', 'inhibitory']:
        sub = df[df.response_type == resp_type].copy()
        if sub.empty:
            print(f'No {resp_type} responses.')
            continue

        color = RESP_COLOR[resp_type]
        n_blk = len(BLOCKER_ORDER)
        fig, axes = plt.subplots(
            n_blk, 2,
            figsize=(14, 3.5 * n_blk),
            constrained_layout=True,
        )

        for row_i, blocker in enumerate(BLOCKER_ORDER):
            sub_b = sub[sub.blocker == blocker]

            # Left: x = PRR, pooled across powers
            ax = axes[row_i, 0]
            sns.boxplot(data=sub_b, x='prr_str', y='latency_ms',
                        order=prr_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='prr_str', y='latency_ms',
                          order=prr_order_str, ax=ax,
                          hue='power_str', hue_order=power_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel(f'{BLOCKER_LABEL.get(blocker, blocker)}\nLatency (ms)', fontsize=9)
            ax.set_xlabel('PRR' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By PRR  (colour = power)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='Power', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

            # Right: x = power, pooled across PRRs
            ax = axes[row_i, 1]
            sns.boxplot(data=sub_b, x='power_str', y='latency_ms',
                        order=power_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='power_str', y='latency_ms',
                          order=power_order_str, ax=ax,
                          hue='prr_str', hue_order=prr_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel('')
            ax.set_xlabel('Laser power' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By laser power  (colour = PRR)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='PRR', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

        fig.suptitle(
            f'Latency — {resp_type.capitalize()} responses  '
            f'(n={len(sub)}, all sessions pooled)',
            fontsize=13, fontweight='bold',
        )
        fname = figure_dir / f'summary_latency_{resp_type}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()


## Firing rate — by PRR and laser power, per blocker condition
Mean firing rate in the post-stimulus window (`MEAN_WINDOW_MS`) for **significant** responses, pooled across all sessions.  
Layout mirrors the latency section.


In [ ]:
if df.empty:
    print('No significant responses — skipping firing rate plots.')
else:
    for resp_type in ['excitatory', 'inhibitory']:
        sub = df[df.response_type == resp_type].copy()
        if sub.empty:
            print(f'No {resp_type} responses.')
            continue

        color = RESP_COLOR[resp_type]
        n_blk = len(BLOCKER_ORDER)
        fig, axes = plt.subplots(
            n_blk, 2,
            figsize=(14, 3.5 * n_blk),
            constrained_layout=True,
        )

        for row_i, blocker in enumerate(BLOCKER_ORDER):
            sub_b = sub[sub.blocker == blocker]

            # Left: x = PRR
            ax = axes[row_i, 0]
            sns.boxplot(data=sub_b, x='prr_str', y='mean_post_rate',
                        order=prr_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='prr_str', y='mean_post_rate',
                          order=prr_order_str, ax=ax,
                          hue='power_str', hue_order=power_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel(
                f'{BLOCKER_LABEL.get(blocker, blocker)}\nMean firing rate (Hz)',
                fontsize=9,
            )
            ax.set_xlabel('PRR' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By PRR  (colour = power)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='Power', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

            # Right: x = power
            ax = axes[row_i, 1]
            sns.boxplot(data=sub_b, x='power_str', y='mean_post_rate',
                        order=power_order_str, ax=ax,
                        color='lightgray', fliersize=0, linewidth=0.8)
            sns.stripplot(data=sub_b, x='power_str', y='mean_post_rate',
                          order=power_order_str, ax=ax,
                          hue='prr_str', hue_order=prr_order_str,
                          jitter=True, size=3, alpha=0.7,
                          legend=(row_i == 0))
            ax.set_ylabel('')
            ax.set_xlabel('Laser power' if row_i == n_blk - 1 else '', fontsize=8)
            if row_i == 0:
                ax.set_title('By laser power  (colour = PRR)', fontsize=10, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            if row_i == 0 and ax.get_legend():
                ax.legend(title='PRR', fontsize=7, title_fontsize=8,
                          loc='upper right', framealpha=0.8)

        fig.suptitle(
            f'Mean post-stim firing rate — {resp_type.capitalize()} responses  '
            f'(n={len(sub)}, all sessions pooled)',
            fontsize=13, fontweight='bold',
        )
        fname = figure_dir / f'summary_firing_rate_{resp_type}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()


## No-blocker condition — latency & firing rate by PRR × power
One subplot per PRR condition; x-axis = laser power. Each box shows the distribution (median) of significant responses pooled across all sessions.

In [ ]:
from scipy.stats import mannwhitneyu
from itertools import combinations

def _stat_label(p):
    if p >= 0.01:  return '*'
    if p >= 0.001: return '**'
    return '***'

# µW → kPa label mapping
PWR_TO_KPA = {4000: 105, 4500: 153, 5000: 201}

def _pwr_label(pwr):
    kpa = PWR_TO_KPA.get(int(pwr))
    return f'{kpa} kPa' if kpa is not None else f'{int(pwr)} µW'

if df.empty:
    print('No significant responses — skipping.')
else:
    sub_nb = df[
        (df['blocker'] == 'noblocker') & (df['response_type'] == 'excitatory')
    ].copy()
    if sub_nb.empty:
        print('No significant excitatory responses in no-blocker condition.')
    else:
        prr_vals_nb   = sorted(sub_nb['prr'].unique())
        power_vals_nb = sorted(sub_nb['power'].unique())
        n_prr = len(prr_vals_nb)
        n_pow = len(power_vals_nb)

        # ── x positions ───────────────────────────────────────────────────────
        within_spacing = 0.3
        between_extra  = 0.1
        box_width      = 0.25

        positions_flat = []
        group_centers  = {}
        group_edges    = {}
        _x = 0.0
        for prr in prr_vals_nb:
            xs = []
            for pwr in power_vals_nb:
                positions_flat.append((prr, pwr, _x))
                xs.append(_x)
                _x += within_spacing
            group_centers[prr] = np.mean(xs)
            group_edges[prr]   = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
            _x += between_extra

        # ── Figure geometry derived from actual data span ─────────────────────
        tick_positions = [xpos for _, _, xpos in positions_flat]
        tick_labels    = [_pwr_label(pwr) for _, pwr, _ in positions_flat]
        xlim_pad  = box_width * 1.2
        x_span    = (tick_positions[-1] - tick_positions[0]) + 2 * xlim_pad
        fig_width = max(3.5, x_span * 2.0)

        TEAL      = '#5BB5B5'
        TEAL_DARK = '#2E8080'

        for metric, ylabel, fname_tag in [
            ('latency_ms',     'Response latency (ms)', 'latency'),
            ('mean_post_rate', 'Mean firing rate (Hz)', 'firing_rate'),
        ]:
            fig, ax = plt.subplots(figsize=(fig_width, 5))
            fig.subplots_adjust(bottom=0.32)

            np.random.seed(42)
            all_group_data = {prr: {} for prr in prr_vals_nb}

            # ── 1. Draw boxes and strip plots ─────────────────────────────────
            for prr, pwr, xpos in positions_flat:
                data = sub_nb[
                    (sub_nb['prr'] == prr) & (sub_nb['power'] == pwr)
                ][metric].dropna().values

                if len(data) == 0:
                    continue

                all_group_data[prr][xpos] = data

                ax.boxplot(
                    data,
                    positions   =[xpos],
                    widths      =box_width,
                    patch_artist=True,
                    flierprops  =dict(marker='o', markersize=2.5, linestyle='none',
                                      markerfacecolor=TEAL_DARK, markeredgecolor='none'),
                    medianprops =dict(color=TEAL_DARK, linewidth=2),
                    boxprops    =dict(facecolor=TEAL, edgecolor=TEAL_DARK, linewidth=0.8),
                    whiskerprops=dict(color=TEAL_DARK, linewidth=0.8),
                    capprops    =dict(color=TEAL_DARK, linewidth=0.8),
                    showfliers  =True,
                )
                jitter = np.random.uniform(-0.08, 0.08, len(data))
                ax.scatter(xpos + jitter, data,
                           s=10, color=TEAL_DARK, alpha=0.45, zorder=3, linewidths=0)

            # ── 2. Print descriptive stats and run pairwise tests ─────────────
            print(f'\n{"="*60}')
            print(f'  {ylabel}  —  No-blocker, excitatory')
            print(f'{"="*60}')
            for prr in prr_vals_nb:
                print(f'\n  PRR = {int(prr/1000)} kHz')
                for pwr in power_vals_nb:
                    g = sub_nb[
                        (sub_nb['prr'] == prr) & (sub_nb['power'] == pwr)
                    ][metric].dropna().values
                    if len(g) == 0:
                        print(f'    {_pwr_label(pwr)} : no data')
                        continue
                    q1, med, q3 = np.percentile(g, [25, 50, 75])
                    print(f'    {_pwr_label(pwr):>10} : '
                          f'median={med:.2f}  Q1={q1:.2f}  Q3={q3:.2f}  n={len(g)}')

                pos_list  = sorted(all_group_data[prr].keys())
                pwr_order = [pwr for _, pwr, xpos in positions_flat
                             if _ == prr and xpos in pos_list]
                groups    = [all_group_data[prr][p] for p in pos_list]
                pairs     = sorted(combinations(range(len(pos_list)), 2),
                                   key=lambda pr: pr[1] - pr[0])
                print()
                for i, j in pairs:
                    g1, g2 = groups[i], groups[j]
                    if len(g1) < 2 or len(g2) < 2:
                        continue
                    _, p = mannwhitneyu(g1, g2, alternative='two-sided')
                    sig  = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                    print(f'    {_pwr_label(pwr_order[i])} vs {_pwr_label(pwr_order[j])} : '
                          f'p={p:.4f}  {sig}')

            # ── 3. Stat brackets — significant pairs only ─────────────────────
            y_lo, y_hi = ax.get_ylim()
            step     = (y_hi - y_lo) * 0.10
            new_ymax = y_hi

            for prr, group_data in all_group_data.items():
                if len(group_data) < 2:
                    continue

                pos_list = sorted(group_data.keys())
                groups   = [group_data[p] for p in pos_list]
                pairs    = sorted(combinations(range(len(pos_list)), 2),
                                  key=lambda pr: pr[1] - pr[0])

                local_max = max(np.max(g) for g in groups if len(g) > 0)
                top_at    = {k: local_max for k in range(len(pos_list))}

                for i, j in pairs:
                    g1, g2 = groups[i], groups[j]
                    if len(g1) < 2 or len(g2) < 2:
                        continue
                    _, p = mannwhitneyu(g1, g2, alternative='two-sided')

                    if p >= 0.05:
                        continue

                    label     = _stat_label(p)
                    bracket_y = max(top_at[k] for k in range(i, j + 1)) + step * 0.5
                    tip_y     = bracket_y - step * 0.25
                    xi, xj   = pos_list[i], pos_list[j]

                    ax.plot([xi, xi, xj, xj],
                            [tip_y, bracket_y, bracket_y, tip_y],
                            'k-', lw=0.9, clip_on=False)
                    ax.text((xi + xj) / 2, bracket_y + step * 0.05, label,
                            ha='center', va='bottom', fontsize=7.5, clip_on=False,
                            fontweight='bold')

                    for k in range(i, j + 1):
                        top_at[k] = bracket_y + step * 0.45

                new_ymax = max(new_ymax, max(top_at.values()) + step * 0.3)

            ax.set_ylim(y_lo, new_ymax + step * 0.5)

            # ── 4. Tick and group labels ──────────────────────────────────────
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(tick_labels, fontsize=8, rotation=45, ha='right')
            ax.tick_params(axis='x', length=0)

            for prr in prr_vals_nb:
                xc     = group_centers[prr]
                xl, xr = group_edges[prr]

                ax.annotate(
                    f'{int(prr / 1000)} kHz',
                    xy=(xc, 0), xycoords=('data', 'axes fraction'),
                    xytext=(0, -48), textcoords='offset points',
                    ha='center', va='top', fontsize=9, fontweight='bold',
                    annotation_clip=False,
                )
                ax.annotate(
                    '',
                    xy=(xr, 0),     xycoords=('data', 'axes fraction'),
                    xytext=(xl, 0), textcoords=('data', 'axes fraction'),
                    arrowprops=dict(arrowstyle='-', color='k', lw=0.8),
                    annotation_clip=False,
                )

            # ── 5. Styling ────────────────────────────────────────────────────
            ax.set_xlim(tick_positions[0] - xlim_pad, tick_positions[-1] + xlim_pad)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_xlabel('Pulse repetition frequency', fontsize=10, labelpad=38)
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.3)
            ax.set_axisbelow(True)

            fname = figure_dir / f'summary_noblocker_excitatory_{fname_tag}_prr_x_power.png'
            fig.savefig(fname, dpi=150, bbox_inches='tight')
            print(f'\nSaved: {fname}')
            plt.show()


## Response counts per cell type — by PRR and laser power, per blocker condition
Includes **all** cells (responders and non-responders) from `zscore_store_all`.  
One overall figure (majority vote across PRR/power) + one figure per blocker.


In [ ]:
if not HAS_TYPING:
    print('Cell typing data not available — skipping.')
else:
    # Build full table including non-responders
    _rows_all = []
    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        resp = d['response_type'] if d['significant'] else 'none'
        _rows_all.append(dict(cluster_id=cid, blocker=blk,
                              prr=float(prr), power=float(pwr),
                              response_type=resp,
                              session=d['session']))
    df_all = pd.DataFrame(_rows_all)
    df_all['cell_type'] = df_all['cluster_id'].map(lambda c: cid_to_ct.get(c, 'Unknown'))
    df_all['resp_label'] = df_all['response_type'].map({
        'excitatory': 'Excitatory',
        'inhibitory': 'Inhibitory',
        'none':       'No response',
    })
    CT_PRESENT = [ct for ct in CT_ORDER if ct in df_all['cell_type'].unique()]
    RESP_LABELS_PLOT = ['Excitatory', 'Inhibitory', 'No response']
    RESP_COLORS_PLOT = {
        'Excitatory':  '#E63946',
        'Inhibitory':  '#457B9D',
        'No response': '#CCCCCC',
    }

    def _count_table_multi(sub):
        ct = (
            sub.groupby(['cell_type', 'resp_label'])
            .size()
            .unstack(fill_value=0)
            .reindex(index=CT_PRESENT, columns=RESP_LABELS_PLOT, fill_value=0)
        )
        ct  = ct.loc[ct.sum(axis=1) > 0]
        pct = ct.div(ct.sum(axis=1), axis=0) * 100
        return ct, pct

    def _plot_ct_panel(ax, counts, pct, title, show_ylabel=False):
        x       = np.arange(len(counts.index))
        n_resp  = len(RESP_LABELS_PLOT)
        width   = 0.22
        offsets = np.linspace(-(n_resp - 1) / 2, (n_resp - 1) / 2, n_resp) * width
        for resp, offset in zip(RESP_LABELS_PLOT, offsets):
            heights = [int(counts.loc[ct, resp]) if ct in counts.index else 0
                       for ct in counts.index]
            pcts    = [pct.loc[ct, resp]          if ct in pct.index    else 0.0
                       for ct in counts.index]
            bars = ax.bar(x + offset, heights, width,
                          color=RESP_COLORS_PLOT[resp], edgecolor='k',
                          linewidth=0.5, alpha=0.88, label=resp)
            for bar, h, p in zip(bars, heights, pcts):
                if h > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.05,
                            f'{h}\n({p:.0f}%)',
                            ha='center', va='bottom', fontsize=5.5,
                            fontweight='bold', color='#222')
        ax.set_xticks(x)
        ax.set_xticklabels(list(counts.index), fontsize=8)
        ax.set_xlabel('Cell type', fontsize=8)
        if show_ylabel:
            ax.set_ylabel('N cells', fontsize=8)
        ax.set_title(title, fontsize=8, fontweight='bold')
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)

    legend_handles = [
        mpatches.Patch(facecolor=RESP_COLORS_PLOT[r], edgecolor='k', label=r)
        for r in RESP_LABELS_PLOT
    ]

    # ── Overall figure (majority vote across PRR/power, all sessions) ─────────
    fig_ov, axes_ov = plt.subplots(
        len(BLOCKER_ORDER), 1,
        figsize=(7, 3.5 * len(BLOCKER_ORDER)),
        constrained_layout=True,
    )
    for ax_ov, blocker in zip(axes_ov, BLOCKER_ORDER):
        sub_b = df_all[df_all['blocker'] == blocker]
        if sub_b.empty:
            ax_ov.set_visible(False)
            continue
        majority = (
            sub_b.groupby(['cluster_id', 'cell_type'])['resp_label']
            .agg(lambda s: s.mode()[0])
            .reset_index()
        )
        counts_ov, pct_ov = _count_table_multi(majority)
        n_cells = sub_b['cluster_id'].nunique()
        _plot_ct_panel(ax_ov, counts_ov, pct_ov,
                       f'{BLOCKER_LABEL.get(blocker, blocker)}  (n={n_cells} cells, majority vote)',
                       show_ylabel=True)
    fig_ov.legend(handles=legend_handles, title='Response type',
                  loc='lower center', ncol=len(RESP_LABELS_PLOT),
                  bbox_to_anchor=(0.5, -0.02), fontsize=8, title_fontsize=9, framealpha=0.85)
    fig_ov.suptitle('Response type × cell type — overall (all sessions pooled)',
                    fontsize=12, fontweight='bold')
    fname = figure_dir / 'summary_cell_type_counts_overall.png'
    fig_ov.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()

    # ── Per-blocker figures: rows = PRR, cols = power ──────────────────────────
    prr_vals_all   = sorted(df_all['prr'].unique())
    power_vals_all = sorted(df_all['power'].unique())

    for blocker in BLOCKER_ORDER:
        sub_b = df_all[df_all['blocker'] == blocker]
        if sub_b.empty:
            continue

        fig, axes = plt.subplots(
            len(prr_vals_all), len(power_vals_all),
            figsize=(4.5 * len(power_vals_all), 3.5 * len(prr_vals_all)),
            constrained_layout=True,
            squeeze=False,
        )
        for row_i, prr in enumerate(prr_vals_all):
            for col_i, power in enumerate(power_vals_all):
                ax  = axes[row_i, col_i]
                sub = sub_b[(sub_b['prr'] == prr) & (sub_b['power'] == power)]
                if sub.empty:
                    ax.set_visible(False)
                    continue
                counts, pct = _count_table_multi(sub)
                _plot_ct_panel(ax, counts, pct,
                               f'PRR={int(prr/1000)} kHz | {int(power)} µW',
                               show_ylabel=(col_i == 0))

        fig.legend(handles=legend_handles, title='Response type',
                   loc='lower center', ncol=len(RESP_LABELS_PLOT),
                   bbox_to_anchor=(0.5, -0.04), fontsize=8, title_fontsize=9, framealpha=0.85)
        fig.suptitle(
            f'Response type × cell type — {BLOCKER_LABEL.get(blocker, blocker)}  (all sessions pooled)',
            fontsize=12, fontweight='bold',
        )
        fname = figure_dir / f'summary_cell_type_counts_{blocker.replace(" ", "_")}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()


## Which precise cell types respond to PA?
Using the **full Baden type name** (e.g. "OFF alpha", "ON sustained 1").  
Shows the number of cells with ≥1 significant PA response across all conditions, broken down by full type and by session.


In [ ]:
if not HAS_TYPING:
    print('Cell typing data not available — skipping.')
elif df.empty:
    print('No significant responses — skipping.')
else:
    # Per-cell summary: does this cell respond in any condition?
    df_full = df.copy()
    df_full['full_type'] = df_full['cluster_id'].map(lambda c: cid_to_fulltype.get(c, 'Unknown'))

    # Count unique responding cells per full Baden type (collapsed across conditions)
    responders_by_type = (
        df_full.groupby(['full_type', 'session'])['cluster_id']
        .nunique()
        .unstack(fill_value=0)
    )
    # Sort by total across sessions
    responders_by_type['_total'] = responders_by_type.sum(axis=1)
    responders_by_type = responders_by_type.sort_values('_total', ascending=False)
    responders_by_type = responders_by_type.drop(columns='_total')

    # ── Print summary table ───────────────────────────────────────────────────
    print('Unique responding cells per Baden type (any condition):')
    print(responders_by_type.to_string())

    # ── Bar chart ─────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(max(10, len(responders_by_type) * 0.6), 5),
                           constrained_layout=True)
    bottom = np.zeros(len(responders_by_type))
    for i, session_id in enumerate(sorted(df['session'].unique())):
        col = responders_by_type.get(session_id,
                                     pd.Series(0, index=responders_by_type.index))
        ax.bar(np.arange(len(responders_by_type)), col.values,
               bottom=bottom,
               color=SESSION_COLORS_MAP.get(session_id, f'C{i}'),
               label=session_id, edgecolor='k', linewidth=0.4, alpha=0.85)
        bottom += col.values

    ax.set_xticks(np.arange(len(responders_by_type)))
    ax.set_xticklabels(responders_by_type.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('N responding cells', fontsize=10)
    ax.set_xlabel('Baden cell type', fontsize=10)
    ax.set_title('Cells responding to PA — by precise Baden type  (any condition, all sessions)',
                 fontsize=11, fontweight='bold')
    ax.legend(title='Session', fontsize=8, title_fontsize=9, framealpha=0.85)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
    fname = figure_dir / 'summary_responding_cells_by_precise_type.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'\nSaved: {fname}')
    plt.show()

    # ── Also show excitatory vs inhibitory breakdown ───────────────────────────
    fig2, axes2 = plt.subplots(1, 2, figsize=(max(12, len(responders_by_type) * 1.2), 5),
                               constrained_layout=True)
    all_types = responders_by_type.index.tolist()

    for ax2, resp_type in zip(axes2, ['excitatory', 'inhibitory']):
        sub = df_full[df_full.response_type == resp_type]
        if sub.empty:
            ax2.set_visible(False)
            continue
        counts_rt = (
            sub.groupby(['full_type', 'session'])['cluster_id']
            .nunique()
            .unstack(fill_value=0)
            .reindex(index=all_types, fill_value=0)
        )
        bottom2 = np.zeros(len(all_types))
        for i, session_id in enumerate(sorted(df['session'].unique())):
            col = counts_rt.get(session_id, pd.Series(0, index=all_types))
            ax2.bar(np.arange(len(all_types)), col.values,
                    bottom=bottom2,
                    color=SESSION_COLORS_MAP.get(session_id, f'C{i}'),
                    label=session_id, edgecolor='k', linewidth=0.4, alpha=0.85)
            bottom2 += col.values

        ax2.set_xticks(np.arange(len(all_types)))
        ax2.set_xticklabels(all_types, rotation=45, ha='right', fontsize=8)
        ax2.set_ylabel('N responding cells', fontsize=9)
        ax2.set_title(f'{resp_type.capitalize()} responders', fontsize=10, fontweight='bold')
        ax2.spines[['top', 'right']].set_visible(False)
        ax2.grid(axis='y', alpha=0.3, linestyle='--')
        ax2.set_axisbelow(True)
        ax2.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax2.legend(title='Session', fontsize=7, title_fontsize=8, framealpha=0.85)

    fig2.suptitle('PA-responding cells by precise Baden type and response polarity',
                  fontsize=11, fontweight='bold')
    fname2 = figure_dir / 'summary_responding_cells_by_type_exc_inh.png'
    fig2.savefig(fname2, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname2}')
    plt.show()


## Response evolution — cells responding under no-blocker AND ACET / ACET+LAP4
Filters to cells that have ≥1 significant response in:
- **No blocker** condition, AND
- **ACET** or **ACET + LAP4** condition.

For those cells, the response type in each blocker condition is shown as a stacked bar (majority vote across PRR/power).


In [ ]:
if df.empty:
    print('No significant responses — skipping.')
else:
    # ── Identify cells responding in noblocker ─────────────────────────────────
    nb_responders = set(
        df[df.blocker == 'noblocker']['cluster_id'].unique()
    )
    # Identify cells responding in acet OR acet lap4
    acet_responders = set(
        df[df.blocker.isin(['acet', 'acet lap4'])]['cluster_id'].unique()
    )
    # Intersection
    target_cells = nb_responders & acet_responders
    print(f'Cells responding in no-blocker: {len(nb_responders)}')
    print(f'Cells responding in ACET or ACET+LAP4: {len(acet_responders)}')
    print(f'Intersection (no-blocker AND ACET/LAP4): {len(target_cells)}')

    if not target_cells:
        print('No cells in intersection — skipping.')
    else:
        # Build full response table for target cells
        _rows_tc = []
        for (cid, blk, prr, pwr), d in zscore_store_all.items():
            if cid not in target_cells:
                continue
            resp = d['response_type'] if d['significant'] else 'none'
            _rows_tc.append(dict(
                cluster_id=cid, blocker=blk,
                prr=float(prr), power=float(pwr),
                response_type=resp,
                resp_label={'excitatory': 'Excitatory',
                            'inhibitory': 'Inhibitory',
                            'none':       'No response'}.get(resp, resp),
                session=d['session'],
            ))
        df_tc = pd.DataFrame(_rows_tc)
        df_tc['cell_type'] = df_tc['cluster_id'].map(lambda c: cid_to_ct.get(c, 'Unknown'))

        RESP_LABELS_PLOT = ['Excitatory', 'Inhibitory', 'No response']
        RESP_COLORS_PLOT = {
            'Excitatory':  '#E63946',
            'Inhibitory':  '#457B9D',
            'No response': '#CCCCCC',
        }

        # ── 1. Overall stacked bar: response type counts per blocker ───────────
        majority_per_cell = (
            df_tc.groupby(['cluster_id', 'blocker'])['resp_label']
            .agg(lambda s: s.mode()[0])
            .reset_index()
        )
        counts_per_blk = (
            majority_per_cell.groupby(['blocker', 'resp_label'])
            .size()
            .unstack(fill_value=0)
            .reindex(index=BLOCKER_ORDER, columns=RESP_LABELS_PLOT, fill_value=0)
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

        # Left: stacked bar by blocker
        ax = axes[0]
        bottom = np.zeros(len(BLOCKER_ORDER))
        for resp in RESP_LABELS_PLOT:
            vals = counts_per_blk[resp].values
            bars = ax.bar(
                np.arange(len(BLOCKER_ORDER)), vals,
                bottom=bottom,
                color=RESP_COLORS_PLOT[resp],
                label=resp, edgecolor='k', linewidth=0.5, alpha=0.88,
            )
            for bar, v, b in zip(bars, vals, bottom):
                if v > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, b + v / 2,
                            str(v), ha='center', va='center',
                            fontsize=9, fontweight='bold', color='k')
            bottom += vals

        ax.set_xticks(np.arange(len(BLOCKER_ORDER)))
        ax.set_xticklabels([BLOCKER_LABEL.get(b, b) for b in BLOCKER_ORDER],
                           rotation=20, ha='right', fontsize=9)
        ax.set_ylabel('N cells (majority vote)', fontsize=10)
        ax.set_title(
            f'Response type per blocker condition\n'
            f'(n={len(target_cells)} cells: no-blocker AND ACET/LAP4 responders)',
            fontsize=10, fontweight='bold',
        )
        ax.legend(fontsize=8, framealpha=0.85)
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

        # Right: per-cell heatmap (cluster × blocker → response type)
        ax2 = axes[1]
        # Encode response type as integer for colouring
        RESP_INT = {'Excitatory': 1, 'Inhibitory': -1, 'No response': 0}
        pivot = (
            majority_per_cell.pivot_table(
                index='cluster_id', columns='blocker',
                values='resp_label', aggfunc=lambda x: x.iloc[0],
            )
            .reindex(columns=BLOCKER_ORDER)
        )
        pivot_int = pivot.applymap(lambda x: RESP_INT.get(x, 0) if isinstance(x, str) else 0)
        cmap = plt.cm.get_cmap('RdBu', 3)
        im = ax2.imshow(pivot_int.values, aspect='auto', cmap=cmap,
                        vmin=-1.5, vmax=1.5, interpolation='nearest')
        ax2.set_xticks(np.arange(len(BLOCKER_ORDER)))
        ax2.set_xticklabels([BLOCKER_LABEL.get(b, b) for b in BLOCKER_ORDER],
                             rotation=25, ha='right', fontsize=8)
        ax2.set_yticks(np.arange(len(pivot_int)))
        ax2.set_yticklabels(pivot_int.index, fontsize=6)
        ax2.set_ylabel('Cell ID', fontsize=9)
        ax2.set_title('Per-cell response type across blockers', fontsize=10, fontweight='bold')
        cbar = plt.colorbar(im, ax=ax2, ticks=[-1, 0, 1])
        cbar.ax.set_yticklabels(['Inhibitory', 'No response', 'Excitatory'], fontsize=7)

        fname = figure_dir / 'summary_target_cells_blocker_comparison.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'Saved: {fname}')
        plt.show()

        # ── 2. Breakdown by cell type ──────────────────────────────────────────
        if HAS_TYPING:
            fig3, axes3 = plt.subplots(
                1, len(BLOCKER_ORDER),
                figsize=(4.5 * len(BLOCKER_ORDER), 5),
                constrained_layout=True,
            )
            CT_PRESENT_TC = [ct for ct in CT_ORDER if ct in df_tc['cell_type'].unique()]

            for ax3, blocker in zip(axes3, BLOCKER_ORDER):
                sub_b = df_tc[df_tc['blocker'] == blocker]
                if sub_b.empty:
                    ax3.set_visible(False)
                    continue
                majority_b = (
                    sub_b.groupby(['cluster_id', 'cell_type'])['resp_label']
                    .agg(lambda s: s.mode()[0])
                    .reset_index()
                )
                counts_b = (
                    majority_b.groupby(['cell_type', 'resp_label'])
                    .size()
                    .unstack(fill_value=0)
                    .reindex(index=CT_PRESENT_TC, columns=RESP_LABELS_PLOT, fill_value=0)
                )
                bottom3 = np.zeros(len(CT_PRESENT_TC))
                for resp in RESP_LABELS_PLOT:
                    vals3 = counts_b.get(resp, pd.Series(0, index=CT_PRESENT_TC)).values
                    ax3.bar(
                        np.arange(len(CT_PRESENT_TC)), vals3,
                        bottom=bottom3,
                        color=RESP_COLORS_PLOT[resp],
                        label=resp, edgecolor='k', linewidth=0.5, alpha=0.88,
                    )
                    bottom3 += vals3

                ax3.set_xticks(np.arange(len(CT_PRESENT_TC)))
                ax3.set_xticklabels(CT_PRESENT_TC, fontsize=9)
                ax3.set_xlabel('Cell type', fontsize=9)
                ax3.set_ylabel('N cells', fontsize=9)
                ax3.set_title(BLOCKER_LABEL.get(blocker, blocker),
                              fontsize=10, fontweight='bold')
                ax3.spines[['top', 'right']].set_visible(False)
                ax3.grid(axis='y', alpha=0.3)
                ax3.set_axisbelow(True)
                ax3.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
                if blocker == BLOCKER_ORDER[0]:
                    ax3.legend(fontsize=8, framealpha=0.85)

            fig3.suptitle(
                f'Response type per blocker × cell type\n'
                f'(n={len(target_cells)} cells: no-blocker AND ACET/LAP4 responders)',
                fontsize=11, fontweight='bold',
            )
            fname3 = figure_dir / 'summary_target_cells_by_celltype.png'
            fig3.savefig(fname3, dpi=150, bbox_inches='tight')
            print(f'Saved: {fname3}')
            plt.show()


## PSTHs — target cells (no-blocker AND ACET / ACET+LAP4 responders)
One figure per cell; layout: **rows = PRR**, **columns = laser power**.  
Each panel overlays the smoothed firing rate for all four blocker conditions (colour-coded).  
Solid lines = significant response; faded lines = non-significant.  
Dashed vertical marker = response latency (only for significant responses).  
Dotted horizontal lines = excitatory / inhibitory threshold from the **no-blocker** baseline.


In [ ]:
if df.empty or not target_cells:
    print('No target cells — skipping PSTH plots.')
else:
    BLOCKER_COLORS = BLK_COLOR

    # Collect all PRR / power present across target cells
    _tc_keys  = [(cid, blk, prr, pwr) for (cid, blk, prr, pwr) in zscore_store_all
                 if cid in target_cells]
    all_prr   = sorted({k[2] for k in _tc_keys})
    all_power = sorted({k[3] for k in _tc_keys})

    out_dir_psth = figure_dir / 'target_cell_psths'
    os.makedirs(out_dir_psth, exist_ok=True)

    for cluster_id in tqdm(sorted(target_cells), desc='Plotting PSTHs'):
        cell_type = cid_to_ct.get(cluster_id, 'Unknown')
        full_type = cid_to_fulltype.get(cluster_id, 'Unknown')
        # Recover session from any zscore_store_all entry for this cell
        _ses = next(
            (d['session'] for (cid, _, _, _), d in zscore_store_all.items()
             if cid == cluster_id),
            '?'
        )

        # PRR / power values that actually have data for this cell
        cell_prr   = sorted({k[2] for k in _tc_keys if k[0] == cluster_id})
        cell_power = sorted({k[3] for k in _tc_keys if k[0] == cluster_id})
        if not cell_prr or not cell_power:
            continue

        fig, axes = plt.subplots(
            len(cell_prr), len(cell_power),
            figsize=(4.5 * len(cell_power), 3.5 * len(cell_prr)),
            squeeze=False,
            constrained_layout=True,
        )

        for row_i, prr in enumerate(cell_prr):
            for col_i, power in enumerate(cell_power):
                ax = axes[row_i, col_i]

                # Reference baseline from no-blocker condition
                nb_key = (cluster_id, 'noblocker', prr, power)
                mu_ref = sd_ref = bd_ms = None
                if nb_key in zscore_store_all:
                    _d = zscore_store_all[nb_key]
                    mu_ref = _d['mu']
                    sd_ref = _d['sd']
                    bd_ms  = _d['bd'] * 1000  # burst duration in ms

                has_any = False
                for blocker in BLOCKER_ORDER:
                    key = (cluster_id, blocker, prr, power)
                    if key not in zscore_store_all:
                        continue
                    d = zscore_store_all[key]
                    has_any = True

                    if bd_ms is None:
                        bd_ms = d['bd'] * 1000

                    color = BLOCKER_COLORS[blocker]
                    sig   = d['significant']
                    alpha = 1.0  if sig else 0.30
                    lw    = 1.8  if sig else 1.0
                    zord  = 3    if sig else 1

                    ax.plot(t_centers, d['rate'],
                            color=color, linewidth=lw, alpha=alpha, zorder=zord,
                            label=BLOCKER_LABEL.get(blocker, blocker))

                    if sig and not np.isnan(d['latency']):
                        ax.axvline(d['latency'], color=color,
                                   linewidth=1.2, linestyle='--',
                                   alpha=0.85, zorder=zord)

                if not has_any:
                    ax.set_visible(False)
                    continue

                # Threshold reference lines (from no-blocker baseline)
                if mu_ref is not None and sd_ref is not None:
                    ax.axhline(mu_ref,
                               color='#555', ls='--', lw=0.8, alpha=0.5, zorder=0)
                    ax.axhline(mu_ref + K_SD_EXCIT * sd_ref,
                               color=RESP_COLOR['excitatory'],
                               ls=':', lw=0.9, alpha=0.5, zorder=0)
                    ax.axhline(max(MIN_INHIB_THRESHOLD, mu_ref - K_SD_INHIB * sd_ref),
                               color=RESP_COLOR['inhibitory'],
                               ls=':', lw=0.9, alpha=0.5, zorder=0)

                # Stimulus window shading + onset line
                ax.axvline(0, color='k', ls='--', lw=0.9, alpha=0.55, zorder=0)
                if bd_ms is not None and bd_ms > 0:
                    ax.axvspan(0, bd_ms, color='gold', alpha=0.12, zorder=0)
                ax.axvspan(-PRE_TIME_MS, 0, color='lightcyan', alpha=0.12, zorder=0)

                ax.set_xlim(-PRE_TIME_MS, POST_TIME_MS)
                ax.set_title(f'PRR = {int(prr/1000)} kHz  |  {int(power)} µW',
                             fontsize=9, fontweight='bold')
                if row_i == len(cell_prr) - 1:
                    ax.set_xlabel('Time (ms)', fontsize=8)
                if col_i == 0:
                    ax.set_ylabel('Firing rate (Hz)', fontsize=8)
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(alpha=0.18)
                ax.tick_params(labelsize=7)

        # Legend: blocker colours
        _blockers_present = [
            b for b in BLOCKER_ORDER
            if any((cluster_id, b, prr, pwr) in zscore_store_all
                   for prr in cell_prr for pwr in cell_power)
        ]
        legend_lines = [
            Line2D([0], [0], color=BLOCKER_COLORS[b], lw=2,
                   label=BLOCKER_LABEL.get(b, b))
            for b in _blockers_present
        ] + [
            Line2D([0], [0], color='#555',                 lw=1.2, ls='--',
                   alpha=0.7, label='Baseline (no-blocker)'),
            Line2D([0], [0], color=RESP_COLOR['excitatory'], lw=1.0, ls=':',
                   alpha=0.7, label='Excit. threshold'),
            Line2D([0], [0], color=RESP_COLOR['inhibitory'], lw=1.0, ls=':',
                   alpha=0.7, label='Inhib. threshold'),
        ]
        fig.legend(handles=legend_lines,
                   loc='lower center', ncol=len(legend_lines),
                   bbox_to_anchor=(0.5, -0.06),
                   fontsize=8, framealpha=0.85)

        fig.suptitle(
            f'{cluster_id}  ·  {full_type}  ·  {_ses}',
            fontsize=11, fontweight='bold',
        )
        fname = out_dir_psth / f'psth_{cluster_id}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {len(target_cells)} PSTH figure(s) → {out_dir_psth}')


## Responsive cells per blocker condition — PRR 5000 Hz × Power 5000 µW
Bars grouped by ON / OFF / ON-OFF; colour = blocker condition. Bar height = total unique responsive cells pooled across sessions.

**Statistics:**
- **Fisher's exact test** (two-sided, pooled across sessions) — shown as brackets on the plot.
- **GEE logistic regression** (`responsive ~ blocker`, session as grouping variable, exchangeable correlation) — printed as OR [95 % CI] and p-value. Note: robust SE from GEE can be conservative with only 5 sessions.

In [ ]:
from scipy.stats import fisher_exact as _fisher_exact
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable

BLOCKERS_PLOT   = ['noblocker', 'acet', 'acet lap4']
CT_PLOT         = ['ON', 'OFF', 'ON-OFF']
BLOCKER_COLORS2 = BLK_COLOR
BLOCKER_LABEL_SHORT = {'noblocker': 'No blk', 'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}
PAIRS_IDX       = sorted([(0, 1), (0, 2), (1, 2)], key=lambda p: p[1] - p[0])

# ── Binary response table: one row per cell × blocker condition ───────────────
_rows_bin = []
for (cid, blk, prr, pwr), d in zscore_store_all.items():
    if prr != 5000.0 or pwr != 5000.0:
        continue
    if blk not in BLOCKERS_PLOT:
        continue
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_PLOT:
        continue
    _rows_bin.append({
        'cluster_id': cid,
        'session':    d['session'],
        'blocker':    blk,
        'cell_type':  ct,
        'excitatory': int(d['significant'] and d['response_type'] == 'excitatory'),
        'inhibitory': int(d['significant'] and d['response_type'] == 'inhibitory'),
    })
df_bin = pd.DataFrame(_rows_bin)

if df_bin.empty:
    print('No data for prr=5000, power=5000 in zscore_store_all.')
else:
    sessions = sorted(df_bin['session'].unique())
    n_sess   = len(sessions)
    print(f'Binary response table: {len(df_bin)} cell–condition rows, {n_sess} sessions')

    def _sig_label(p):
        if p < 0.001: return '***'
        if p < 0.01:  return '**'
        if p < 0.05:  return '*'
        return None

    fisher_pvals = {}   # (resp_type, ct, b1, b2) → p
    gee_pvals    = {}   # (resp_type, ct, b1, b2) → (OR, ci_lo, ci_hi, p)

    for resp_type in ['excitatory', 'inhibitory']:
        print(f'\n{"="*62}')
        print(f'  {resp_type.upper()}')
        print(f'{"="*62}')

        # ── Fisher's exact test ───────────────────────────────────────────────
        print('\n  Fisher\'s exact test (two-sided, pooled across sessions)')
        print(f'  {"Cell type":<10} {"Comparison":<38} {"Counts":>14}  {"p":>7}  Sig')
        print(f'  {"-"*82}')
        for ct in CT_PLOT:
            sub_ct = df_bin[df_bin.cell_type == ct]
            for (i, j) in PAIRS_IDX:
                b1, b2 = BLOCKERS_PLOT[i], BLOCKERS_PLOT[j]
                s1 = sub_ct[sub_ct.blocker == b1][resp_type]
                s2 = sub_ct[sub_ct.blocker == b2][resp_type]
                r1, n1 = int(s1.sum()), len(s1)
                r2, n2 = int(s2.sum()), len(s2)
                if n1 == 0 or n2 == 0:
                    continue
                _, p = _fisher_exact([[r1, n1 - r1], [r2, n2 - r2]],
                                     alternative='two-sided')
                fisher_pvals[(resp_type, ct, b1, b2)] = p
                sig = _sig_label(p) or 'ns'
                comparison = f'{BLOCKER_LABEL[b1]} vs {BLOCKER_LABEL[b2]}'
                counts     = f'{r1}/{n1} vs {r2}/{n2}'
                print(f'  {ct:<10} {comparison:<38} {counts:>14}  {p:>7.4f}  {sig}')

        # ── GEE logistic regression ───────────────────────────────────────────
        print(f'\n  GEE logistic regression (session as grouping, exchangeable)')
        print(f'  {"Cell type":<10} {"Comparison":<38} {"OR [95% CI]":>22}  {"p":>7}  Sig')
        print(f'  {"-"*90}')
        for ct in CT_PLOT:
            sub_ct = df_bin[df_bin.cell_type == ct].copy()
            if sub_ct[resp_type].nunique() < 2:
                print(f'  {ct:<10} no variation in outcome — skipping')
                continue
            for (i, j) in PAIRS_IDX:
                b1, b2   = BLOCKERS_PLOT[i], BLOCKERS_PLOT[j]
                sub_pair = sub_ct[sub_ct.blocker.isin([b1, b2])].copy()
                if len(sub_pair) == 0 or sub_pair[resp_type].nunique() < 2:
                    continue
                sub_pair['_blk'] = (sub_pair['blocker'] == b2).astype(float)
                try:
                    res = smf.gee(
                        f'{resp_type} ~ _blk',
                        groups='session',
                        data=sub_pair,
                        family=Binomial(),
                        cov_struct=Exchangeable(),
                    ).fit(disp=False)
                    p     = float(res.pvalues['_blk'])
                    or_   = float(np.exp(res.params['_blk']))
                    ci    = res.conf_int()
                    ci_lo = float(np.exp(ci.loc['_blk', 0]))
                    ci_hi = float(np.exp(ci.loc['_blk', 1]))
                    gee_pvals[(resp_type, ct, b1, b2)] = (or_, ci_lo, ci_hi, p)
                    sig        = _sig_label(p) or 'ns'
                    comparison = f'{BLOCKER_LABEL[b1]} vs {BLOCKER_LABEL[b2]}'
                    or_str     = f'{or_:.2f} [{ci_lo:.2f}, {ci_hi:.2f}]'
                    print(f'  {ct:<10} {comparison:<38} {or_str:>22}  {p:>7.4f}  {sig}')
                except Exception as e:
                    comparison = f'{BLOCKER_LABEL[b1]} vs {BLOCKER_LABEL[b2]}'
                    print(f'  {ct:<10} {comparison:<38} GEE failed: {e}')

    # ── x-axis layout (mirrors cell 436e9c39) ─────────────────────────────────
    within_spacing = 0.3
    between_extra  = 0.1
    box_width      = 0.25

    positions_flat = []   # (ct, blocker, xpos)
    group_centers  = {}   # ct → centre xpos
    group_edges    = {}   # ct → (left_edge, right_edge)

    _x = 0.0
    for ct in CT_PLOT:
        xs = []
        for blocker in BLOCKERS_PLOT:
            positions_flat.append((ct, blocker, _x))
            xs.append(_x)
            _x += within_spacing
        group_centers[ct] = np.mean(xs)
        group_edges[ct]   = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
        _x += between_extra

    tick_positions = [xpos for _, _, xpos in positions_flat]
    tick_labels    = [BLOCKER_LABEL_SHORT[b] for _, b, _ in positions_flat]
    xlim_pad       = box_width * 1.2
    x_span         = (tick_positions[-1] - tick_positions[0]) + 2 * xlim_pad
    fig_width      = max(3.5, x_span * 2.0)

    # position lookup for brackets: (ct, blocker) → xpos
    pos_lookup = {(ct, b): xp for ct, b, xp in positions_flat}

    # ── Plots ─────────────────────────────────────────────────────────────────
    for resp_type in ['excitatory', 'inhibitory']:
        counts_total = (
            df_bin.groupby(['blocker', 'cell_type'])[resp_type]
            .sum()
            .unstack(fill_value=0)
            .reindex(index=BLOCKERS_PLOT, columns=CT_PLOT, fill_value=0)
        )

        fig, ax = plt.subplots(figsize=(fig_width, 5))
        fig.subplots_adjust(bottom=0.32)

        # ── 1. Bars ───────────────────────────────────────────────────────────
        for ct, blocker, xpos in positions_flat:
            h = int(counts_total.loc[blocker, ct]) if ct in counts_total.columns else 0
            ax.bar(xpos, h, box_width,
                   color=BLOCKER_COLORS2[blocker], edgecolor='k',
                   linewidth=0.6, alpha=0.88)
            if h > 0:
                ax.text(xpos, h + 0.05, str(h),
                        ha='center', va='bottom', fontsize=8, fontweight='bold')

        # ── 2. Statistical brackets (Fisher's exact, significant only) ────────
        y_lo, y_hi = ax.get_ylim()
        step       = (y_hi - y_lo) * 0.10
        new_ymax   = y_hi

        for ct in CT_PLOT:
            top_at = {
                bi: int(counts_total.loc[blocker, ct]) if ct in counts_total.columns else 0
                for bi, blocker in enumerate(BLOCKERS_PLOT)
            }
            for (i, j) in PAIRS_IDX:
                b1, b2 = BLOCKERS_PLOT[i], BLOCKERS_PLOT[j]
                p      = fisher_pvals.get((resp_type, ct, b1, b2))
                if p is None or _sig_label(p) is None:
                    continue
                xi        = pos_lookup[(ct, b1)]
                xj        = pos_lookup[(ct, b2)]
                bracket_y = max(top_at[k] for k in range(i, j + 1)) + step * 0.5
                tip_y     = bracket_y - step * 0.25
                ax.plot([xi, xi, xj, xj],
                        [tip_y, bracket_y, bracket_y, tip_y],
                        'k-', lw=0.9, clip_on=False)
                ax.text((xi + xj) / 2, bracket_y + step * 0.05, _sig_label(p),
                        ha='center', va='bottom', fontsize=7.5,
                        fontweight='bold', clip_on=False)
                for k in range(i, j + 1):
                    top_at[k] = bracket_y + step * 0.45
            new_ymax = max(new_ymax, max(top_at.values()) + step * 0.3)

        ax.set_ylim(y_lo, new_ymax + step * 0.5)

        # ── 3. Tick and group labels ───────────────────────────────────────────
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, fontsize=8, rotation=45, ha='right')
        ax.tick_params(axis='x', length=0)

        for ct in CT_PLOT:
            xc     = group_centers[ct]
            xl, xr = group_edges[ct]
            ax.annotate(
                ct,
                xy=(xc, 0), xycoords=('data', 'axes fraction'),
                xytext=(0, -48), textcoords='offset points',
                ha='center', va='top', fontsize=9, fontweight='bold',
                annotation_clip=False,
            )
            ax.annotate(
                '',
                xy=(xr, 0),     xycoords=('data', 'axes fraction'),
                xytext=(xl, 0), textcoords=('data', 'axes fraction'),
                arrowprops=dict(arrowstyle='-', color='k', lw=0.8),
                annotation_clip=False,
            )

        # ── 4. Legend ─────────────────────────────────────────────────────────
        legend_handles = [
            mpatches.Patch(facecolor=BLOCKER_COLORS2[b], edgecolor='k',
                           label=BLOCKER_LABEL[b], alpha=0.88)
            for b in BLOCKERS_PLOT
        ]
        ax.legend(handles=legend_handles, title='Blocker condition',
                  fontsize=9, title_fontsize=10, framealpha=0.85)

        # ── 5. Styling ────────────────────────────────────────────────────────
        ax.set_xlim(tick_positions[0] - xlim_pad, tick_positions[-1] + xlim_pad)
        ax.set_ylabel('N responsive cells (all sessions pooled)', fontsize=10)
        ax.set_xlabel('Cell type', fontsize=10, labelpad=38)
        ax.set_title(
            f'{resp_type.capitalize()} responses  —  '
            f'PRR = 5000 Hz  |  Power = 5000 µW  (n = {n_sess} sessions)\n'
            f'Brackets: Fisher\'s exact test',
            fontsize=11, fontweight='bold',
        )
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

        fname = figure_dir / f'summary_responsive_cells_prr5000_pwr5000_{resp_type}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        print(f'\nSaved: {fname}')
        plt.show()


## Excitatory vs inhibitory responsive cells — per PRR × power × blocker condition
Two bars per PRR × power position (excitatory / inhibitory); one subplot per blocker condition. All sessions pooled.

In [ ]:
if df.empty:
    print('No significant responses — skipping.')
else:
    BLOCKERS_PLOT3   = ['noblocker', 'acet', 'acet lap4']
    RESP_TYPES_PLOT  = ['excitatory', 'inhibitory']

    prr_vals_all   = sorted(df['prr'].unique())
    power_vals_all = sorted(df['power'].unique())

    # ── x-axis layout (mirrors cell 436e9c39) ─────────────────────────────────
    within_spacing = 0.3
    between_extra  = 0.1
    bar_width      = 0.08
    bar_step       = bar_width + 0.01          # centre-to-centre within a position
    n_blk          = len(BLOCKERS_PLOT3)
    bar_offsets    = np.linspace(-(n_blk - 1) / 2,
                                  (n_blk - 1) / 2,
                                  n_blk) * bar_step   # e.g. [-0.09, 0, 0.09]
    half_span      = abs(bar_offsets[0]) + bar_width / 2   # for group_edges

    positions_flat = []
    group_centers  = {}
    group_edges    = {}

    _x = 0.0
    for prr in prr_vals_all:
        xs = []
        for power in power_vals_all:
            positions_flat.append((prr, power, _x))
            xs.append(_x)
            _x += within_spacing
        group_centers[prr] = np.mean(xs)
        group_edges[prr]   = (xs[0] - half_span, xs[-1] + half_span)
        _x += between_extra

    tick_positions = [xpos for _, _, xpos in positions_flat]
    tick_labels    = [_pwr_label(pwr) for _, pwr, _ in positions_flat]
    xlim_pad  = within_spacing * 0.6
    x_span    = (tick_positions[-1] - tick_positions[0]) + 2 * xlim_pad
    fig_width = max(4, x_span * 2.0)

    fig, axes = plt.subplots(1, 2, figsize=(2 * fig_width, 5))
    fig.subplots_adjust(left=0.08, right=0.97, bottom=0.32, top=0.88, wspace=0.30)

    for ax, resp_type in zip(axes, RESP_TYPES_PLOT):
        sub_r = df[
            df.blocker.isin(BLOCKERS_PLOT3) &
            (df.response_type == resp_type)
        ]

        # ── 1. Bars ───────────────────────────────────────────────────────────
        for prr, power, xpos in positions_flat:
            for blocker, offset in zip(BLOCKERS_PLOT3, bar_offsets):
                n = sub_r[
                    (sub_r.prr == prr) &
                    (sub_r.power == power) &
                    (sub_r.blocker == blocker)
                ]['cluster_id'].nunique()
                x_bar = xpos + offset
                ax.bar(x_bar, n, bar_width,
                       color=BLOCKER_COLORS2[blocker], edgecolor='k',
                       linewidth=0.5, alpha=0.88)
                if n > 0:
                    ax.text(x_bar, n + 0.05, str(n),
                            ha='center', va='bottom', fontsize=7, fontweight='bold')

        # ── 2. Tick and group labels ───────────────────────────────────────────
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, fontsize=8, rotation=45, ha='right')
        ax.tick_params(axis='x', length=0)
        ax.set_xlim(tick_positions[0] - xlim_pad, tick_positions[-1] + xlim_pad)

        for prr in prr_vals_all:
            xc     = group_centers[prr]
            xl, xr = group_edges[prr]
            ax.annotate(
                f'{int(prr / 1000)} kHz',
                xy=(xc, 0), xycoords=('data', 'axes fraction'),
                xytext=(0, -48), textcoords='offset points',
                ha='center', va='top', fontsize=9, fontweight='bold',
                annotation_clip=False,
            )
            ax.annotate(
                '',
                xy=(xr, 0),     xycoords=('data', 'axes fraction'),
                xytext=(xl, 0), textcoords=('data', 'axes fraction'),
                arrowprops=dict(arrowstyle='-', color='k', lw=0.8),
                annotation_clip=False,
            )

        # ── 3. Styling ────────────────────────────────────────────────────────
        ax.set_xlabel('Laser power  (grouped by PRR)', fontsize=10, labelpad=38)
        ax.set_ylabel('N responsive cells (all sessions pooled)', fontsize=10)
        ax.set_title(f'{resp_type.capitalize()} responses',
                     fontsize=11, fontweight='bold')
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.3)
        ax.set_axisbelow(True)
        ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

    legend_handles = [
        mpatches.Patch(facecolor=BLOCKER_COLORS2[b], edgecolor='k',
                       label=BLOCKER_LABEL[b], alpha=0.88)
        for b in BLOCKERS_PLOT3
    ]
    fig.legend(
        handles=legend_handles,
        loc='lower center', ncol=n_blk,
        bbox_to_anchor=(0.5, 0.01),
        fontsize=9, title='Blocker condition', title_fontsize=10, framealpha=0.85,
    )
    fig.suptitle(
        'Excitatory vs inhibitory responsive cells per PRR × power\n(all sessions pooled)',
        fontsize=12, fontweight='bold',
    )

    fname = figure_dir / 'summary_exc_vs_inh_per_prr_power_blocker.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    print(f'Saved: {fname}')
    plt.show()


## Excitatory vs inhibitory per precise Baden type — per blocker × PRR × power
One figure per blocker condition (No blocker / ACET / ACET+LAP4); rows = PRR, columns = laser power.  
Each panel: two bars per Baden type (excitatory in red, inhibitory in blue). Types sorted by total response count. Washout excluded.

In [ ]:
if df.empty or not HAS_TYPING:
    print('No significant responses or typing data not available — skipping.')
else:
    BLOCKERS_PLOT3 = ['noblocker', 'acet', 'acet lap4']

    prr_vals_all   = sorted(df['prr'].unique())
    power_vals_all = sorted(df['power'].unique())

    sub_all = df[
        df.blocker.isin(BLOCKERS_PLOT3) &
        (df['full_type'] != 'Unknown')
    ].copy()

    if sub_all.empty:
        print('No responses with a known precise cell type.')
    else:
        # Global type order: most responsive types first (consistent across all panels)
        type_order = (
            sub_all.groupby('full_type')['cluster_id']
            .nunique()
            .sort_values(ascending=False)
            .index.tolist()
        )

        bar_width = 0.35

        for blocker in BLOCKERS_PLOT3:
            sub_b = sub_all[sub_all.blocker == blocker]
            types_here = [t for t in type_order if t in sub_b['full_type'].values]

            if not types_here:
                print(f'No typed responses for {blocker} — skipping.')
                continue

            n_prr   = len(prr_vals_all)
            n_power = len(power_vals_all)
            n_types = len(types_here)
            x       = np.arange(n_types)

            fig, axes = plt.subplots(
                n_prr, n_power,
                figsize=(max(6, n_types * 0.55) * n_power,
                         4 * n_prr),
                constrained_layout=True,
                squeeze=False,
            )

            for row_i, prr in enumerate(prr_vals_all):
                for col_i, power in enumerate(power_vals_all):
                    ax  = axes[row_i, col_i]
                    sub = sub_b[(sub_b.prr == prr) & (sub_b.power == power)]

                    for resp_type, sign in zip(['excitatory', 'inhibitory'], [-1, 1]):
                        counts = [
                            sub[
                                (sub.full_type == ft) &
                                (sub.response_type == resp_type)
                            ]['cluster_id'].nunique()
                            for ft in types_here
                        ]
                        bars = ax.bar(
                            x + sign * bar_width / 2, counts, bar_width,
                            color=RESP_COLOR[resp_type], edgecolor='k',
                            linewidth=0.5, alpha=0.88,
                        )
                        for bar, c in zip(bars, counts):
                            if c > 0:
                                ax.text(
                                    bar.get_x() + bar.get_width() / 2,
                                    c + 0.05, str(c),
                                    ha='center', va='bottom',
                                    fontsize=7, fontweight='bold',
                                )

                    ax.set_xticks(x)
                    ax.set_xticklabels(types_here, rotation=45, ha='right', fontsize=7.5)
                    ax.set_title(
                        f'PRR = {int(prr / 1000)} kHz  |  {_pwr_label(power)}',
                        fontsize=9, fontweight='bold',
                    )
                    if col_i == 0:
                        ax.set_ylabel('N responsive cells', fontsize=8)
                    ax.spines[['top', 'right']].set_visible(False)
                    ax.grid(axis='y', alpha=0.3)
                    ax.set_axisbelow(True)
                    ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))

            legend_handles = [
                mpatches.Patch(facecolor=RESP_COLOR[r], edgecolor='k',
                               label=r.capitalize(), alpha=0.88)
                for r in ['excitatory', 'inhibitory']
            ]
            fig.legend(
                handles=legend_handles,
                loc='lower center', ncol=2,
                bbox_to_anchor=(0.5, -0.02),
                fontsize=9, title='Response type', title_fontsize=10, framealpha=0.85,
            )
            fig.suptitle(
                f'{BLOCKER_LABEL.get(blocker, blocker)}\n'
                f'Excitatory vs inhibitory per precise Baden type  (all sessions pooled)',
                fontsize=12, fontweight='bold',
            )
            fname = figure_dir / f'summary_exc_inh_precise_type_{blocker.replace(" ", "_")}.png'
            fig.savefig(fname, dpi=150, bbox_inches='tight')
            print(f'Saved: {fname}')
            plt.show()


## Response reversal — excitatory (no blocker) → inhibitory (with blocker)

Identifies cells whose response **flips sign** under the same PRR × power stimulation condition:
- **No blocker**: significant **excitatory** response
- **ACET** or **ACET + LAP4**: significant **inhibitory** response

One PSTH figure per reversal cell; reversal panels are highlighted with a red frame.

In [ ]:
if df.empty:
    print('No significant responses — skipping reversal analysis.')
else:
    BLOCKERS_REVERSAL = ['acet', 'acet lap4']

    # reversal_details: cluster_id -> list of (prr, power, blocker) where reversal occurs
    reversal_details = {}

    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        if blk not in BLOCKERS_REVERSAL:
            continue
        if not (d['significant'] and d['response_type'] == 'inhibitory'):
            continue
        # Check if same cell is excitatory under no-blocker for same prr × power
        nb_key = (cid, 'noblocker', prr, pwr)
        nb_d = zscore_store_all.get(nb_key)
        if nb_d is None:
            continue
        if not (nb_d['significant'] and nb_d['response_type'] == 'excitatory'):
            continue
        reversal_details.setdefault(cid, []).append((prr, pwr, blk))

    reversal_cells = set(reversal_details.keys())

    print(f'Cells with ≥1 excitatory→inhibitory reversal: {len(reversal_cells)}')
    for cid in sorted(reversal_cells):
        ct = cid_to_ct.get(cid, 'Unknown')
        ft = cid_to_fulltype.get(cid, 'Unknown')
        ses = next((d['session'] for (c, _, _, _), d in zscore_store_all.items() if c == cid), '?')
        details = reversal_details[cid]
        print(f'  {cid}  [{ft}]  {ses}')
        for prr, pwr, blk in sorted(details):
            print(f'    PRR={int(prr)} Hz  Power={int(pwr)} µW  blocker={blk}')


In [ ]:
if df.empty or not reversal_cells:
    print('No reversal cells — skipping PSTH plots.')
else:
    _RC_BLOCKER_COLORS = BLK_COLOR

    # Sliding window parameters
    SW_WIN_MS  = 20.0
    SW_STEP_MS = 5.0
    _sw_win_bins  = int(SW_WIN_MS  / BIN_SIZE_MS)   # 40 bins
    _sw_step_bins = int(SW_STEP_MS / BIN_SIZE_MS)   # 10 bins

    def _slide(rate_raw_arr):
        """Apply sliding window average; returns (sw_t, sw_rate)."""
        n = len(rate_raw_arr)
        starts = range(0, n - _sw_win_bins + 1, _sw_step_bins)
        sw_rate = np.array([rate_raw_arr[i:i + _sw_win_bins].mean() for i in starts])
        sw_t    = np.array([t_centers[i + _sw_win_bins // 2]        for i in starts])
        return sw_t, sw_rate

    out_dir_rev = figure_dir / 'reversal_cell_psths'
    os.makedirs(out_dir_rev, exist_ok=True)

    _rev_keys = [(cid, blk, prr, pwr) for (cid, blk, prr, pwr) in zscore_store_all
                 if cid in reversal_cells]

    for cluster_id in tqdm(sorted(reversal_cells), desc='Reversal PSTHs'):
        full_type = cid_to_fulltype.get(cluster_id, 'Unknown')
        _ses = next(
            (d['session'] for (cid, _, _, _), d in zscore_store_all.items() if cid == cluster_id), '?'
        )

        cell_prr   = sorted({k[2] for k in _rev_keys if k[0] == cluster_id})
        cell_power = sorted({k[3] for k in _rev_keys if k[0] == cluster_id})
        if not cell_prr or not cell_power:
            continue

        reversal_panels = {(prr, pwr) for prr, pwr, _ in reversal_details[cluster_id]}

        fig, axes = plt.subplots(
            len(cell_prr), len(cell_power),
            figsize=(4.5 * len(cell_power), 3.5 * len(cell_prr)),
            squeeze=False,
            constrained_layout=True,
        )

        for row_i, prr in enumerate(cell_prr):
            for col_i, power in enumerate(cell_power):
                ax = axes[row_i, col_i]

                nb_key = (cluster_id, 'noblocker', prr, power)
                mu_ref = sd_ref = bd_ms = None
                if nb_key in zscore_store_all:
                    _d = zscore_store_all[nb_key]
                    mu_ref = _d['mu']
                    sd_ref = _d['sd']
                    bd_ms  = _d['bd'] * 1000

                has_any = False
                for blocker in BLOCKER_ORDER:
                    key = (cluster_id, blocker, prr, power)
                    if key not in zscore_store_all:
                        continue
                    d = zscore_store_all[key]
                    has_any = True

                    if bd_ms is None:
                        bd_ms = d['bd'] * 1000

                    sw_t, sw_rate = _slide(d['rate_raw'])

                    color = _RC_BLOCKER_COLORS[blocker]
                    sig   = d['significant']
                    alpha = 1.0 if sig else 0.30
                    lw    = 1.8 if sig else 1.0
                    zord  = 3   if sig else 1

                    ax.plot(sw_t, sw_rate,
                            color=color, linewidth=lw, alpha=alpha, zorder=zord,
                            label=BLOCKER_LABEL.get(blocker, blocker))

                    # Latency from original detection (kept as-is)
                    if sig and not np.isnan(d['latency']):
                        ax.axvline(d['latency'], color=color,
                                   linewidth=1.2, linestyle='--', alpha=0.85, zorder=zord)

                if not has_any:
                    ax.set_visible(False)
                    continue

                if mu_ref is not None and sd_ref is not None:
                    ax.axhline(mu_ref, color='#555', ls='--', lw=0.8, alpha=0.5, zorder=0)
                    ax.axhline(mu_ref + K_SD_EXCIT * sd_ref,
                               color=RESP_COLOR['excitatory'], ls=':', lw=0.9, alpha=0.5, zorder=0)
                    ax.axhline(max(MIN_INHIB_THRESHOLD, mu_ref - K_SD_INHIB * sd_ref),
                               color=RESP_COLOR['inhibitory'], ls=':', lw=0.9, alpha=0.5, zorder=0)

                ax.axvline(0, color='k', ls='--', lw=0.9, alpha=0.55, zorder=0)
                if bd_ms is not None and bd_ms > 0:
                    ax.axvspan(0, bd_ms, color='gold', alpha=0.12, zorder=0)
                ax.axvspan(-PRE_TIME_MS, 0, color='lightcyan', alpha=0.12, zorder=0)

                ax.set_xlim(-PRE_TIME_MS, POST_TIME_MS)

                title_str = f'PRR = {int(prr/1000)} kHz  |  {int(power)} µW'
                is_reversal = (prr, power) in reversal_panels
                title_color = '#C0392B' if is_reversal else 'black'
                reversal_suffix = '  ← REVERSAL' if is_reversal else ''
                ax.set_title(title_str + reversal_suffix,
                             fontsize=9, fontweight='bold', color=title_color)

                if is_reversal:
                    for spine in ax.spines.values():
                        spine.set_edgecolor('#C0392B')
                        spine.set_linewidth(2.0)

                if row_i == len(cell_prr) - 1:
                    ax.set_xlabel('Time (ms)', fontsize=8)
                if col_i == 0:
                    ax.set_ylabel('Firing rate (Hz)', fontsize=8)
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(alpha=0.18)
                ax.tick_params(labelsize=7)

        _blockers_present = [
            b for b in BLOCKER_ORDER
            if any((cluster_id, b, prr, pwr) in zscore_store_all
                   for prr in cell_prr for pwr in cell_power)
        ]
        legend_lines = [
            Line2D([0], [0], color=_RC_BLOCKER_COLORS[b], lw=2,
                   label=BLOCKER_LABEL.get(b, b))
            for b in _blockers_present
        ] + [
            Line2D([0], [0], color='#555', lw=1.2, ls='--', alpha=0.7, label='Baseline (no-blocker)'),
            Line2D([0], [0], color=RESP_COLOR['excitatory'], lw=1.0, ls=':', alpha=0.7, label='Excit. threshold'),
            Line2D([0], [0], color=RESP_COLOR['inhibitory'], lw=1.0, ls=':', alpha=0.7, label='Inhib. threshold'),
        ]
        fig.legend(handles=legend_lines,
                   loc='lower center', ncol=len(legend_lines),
                   bbox_to_anchor=(0.5, -0.06),
                   fontsize=8, framealpha=0.85)

        fig.suptitle(
            f'{cluster_id}  ·  {full_type}  ·  {_ses}  ·  (red = reversal | SW {int(SW_WIN_MS)} ms / {int(SW_STEP_MS)} ms)',
            fontsize=10, fontweight='bold',
        )
        fname = out_dir_rev / f'psth_reversal_{cluster_id}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {len(reversal_cells)} reversal PSTH figure(s) → {out_dir_rev}')


## Latency shift — same response type, different latency with blocker

Identifies cells that respond with the **same polarity** (both excitatory or both inhibitory) under no-blocker and ACET / ACET+LAP4, but whose **response latency shifts** by at least `MIN_LATENCY_DIFF_MS` ms.

One PSTH figure per cell; panels where a latency shift occurs are highlighted with an orange frame.


In [ ]:
if df.empty:
    print('No significant responses — skipping latency-shift analysis.')
else:
    BLOCKERS_LATSHIFT  = ['acet', 'acet lap4']
    MIN_LATENCY_DIFF_MS = 5.0   # ms — minimum shift to be included

    # latshift_details: cid -> [(prr, power, blocker, lat_nb, lat_blk, resp_type), ...]
    latshift_details = {}

    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        if blk not in BLOCKERS_LATSHIFT:
            continue
        if not (d['significant'] and not np.isnan(d['latency'])):
            continue

        nb_key = (cid, 'noblocker', prr, pwr)
        nb_d = zscore_store_all.get(nb_key)
        if nb_d is None:
            continue
        if not (nb_d['significant'] and not np.isnan(nb_d['latency'])):
            continue

        # Same response polarity
        if d['response_type'] != nb_d['response_type']:
            continue

        lat_diff = abs(d['latency'] - nb_d['latency'])
        if lat_diff < MIN_LATENCY_DIFF_MS:
            continue

        latshift_details.setdefault(cid, []).append(
            (prr, pwr, blk, nb_d['latency'], d['latency'], d['response_type'])
        )

    latshift_cells = set(latshift_details.keys())

    print(f'Cells with ≥1 same-type latency shift (≥{MIN_LATENCY_DIFF_MS} ms): {len(latshift_cells)}')
    for cid in sorted(latshift_cells):
        ft  = cid_to_fulltype.get(cid, 'Unknown')
        ses = next((d['session'] for (c, _, _, _), d in zscore_store_all.items() if c == cid), '?')
        print(f'  {cid}  [{ft}]  {ses}')
        for prr, pwr, blk, lat_nb, lat_blk, rtype in sorted(latshift_details[cid]):
            shift = lat_blk - lat_nb
            sign  = '+' if shift >= 0 else ''
            print(f'    PRR={int(prr)} Hz  Power={int(pwr)} µW  blocker={blk}'
                  f'  type={rtype}  lat_nb={lat_nb:.1f} ms → lat_blk={lat_blk:.1f} ms  (Δ={sign}{shift:.1f} ms)')


In [ ]:
if df.empty or not latshift_cells:
    print('No latency-shift cells — skipping PSTH plots.')
else:
    _LS_BLOCKER_COLORS = BLK_COLOR
    _LS_SHIFT_COLOR = '#E67E22'   # orange border for latency-shift panels

    SW_WIN_MS_LS  = 20.0
    SW_STEP_MS_LS = 5.0
    _ls_win_bins  = int(SW_WIN_MS_LS  / BIN_SIZE_MS)
    _ls_step_bins = int(SW_STEP_MS_LS / BIN_SIZE_MS)

    def _slide_ls(rate_raw_arr):
        starts  = range(0, len(rate_raw_arr) - _ls_win_bins + 1, _ls_step_bins)
        sw_rate = np.array([rate_raw_arr[i:i + _ls_win_bins].mean() for i in starts])
        sw_t    = np.array([t_centers[i + _ls_win_bins // 2]        for i in starts])
        return sw_t, sw_rate

    out_dir_ls = figure_dir / 'latshift_cell_psths'
    os.makedirs(out_dir_ls, exist_ok=True)

    _ls_keys = [(cid, blk, prr, pwr) for (cid, blk, prr, pwr) in zscore_store_all
                if cid in latshift_cells]

    for cluster_id in tqdm(sorted(latshift_cells), desc='Latency-shift PSTHs'):
        full_type = cid_to_fulltype.get(cluster_id, 'Unknown')
        _ses = next(
            (d['session'] for (cid, _, _, _), d in zscore_store_all.items() if cid == cluster_id), '?'
        )

        cell_prr   = sorted({k[2] for k in _ls_keys if k[0] == cluster_id})
        cell_power = sorted({k[3] for k in _ls_keys if k[0] == cluster_id})
        if not cell_prr or not cell_power:
            continue

        # (prr, power) panels that have a latency shift for this cell
        shift_panels = {(prr, pwr) for prr, pwr, _, _, _, _ in latshift_details[cluster_id]}
        # Lookup: (prr, power, blocker) -> (lat_nb, lat_blk)
        shift_lookup = {
            (prr, pwr, blk): (lat_nb, lat_blk)
            for prr, pwr, blk, lat_nb, lat_blk, _ in latshift_details[cluster_id]
        }

        fig, axes = plt.subplots(
            len(cell_prr), len(cell_power),
            figsize=(4.5 * len(cell_power), 3.5 * len(cell_prr)),
            squeeze=False,
            constrained_layout=True,
        )

        for row_i, prr in enumerate(cell_prr):
            for col_i, power in enumerate(cell_power):
                ax = axes[row_i, col_i]

                nb_key = (cluster_id, 'noblocker', prr, power)
                mu_ref = sd_ref = bd_ms = None
                if nb_key in zscore_store_all:
                    _d = zscore_store_all[nb_key]
                    mu_ref = _d['mu']
                    sd_ref = _d['sd']
                    bd_ms  = _d['bd'] * 1000

                has_any = False
                for blocker in BLOCKER_ORDER:
                    key = (cluster_id, blocker, prr, power)
                    if key not in zscore_store_all:
                        continue
                    d = zscore_store_all[key]
                    has_any = True

                    if bd_ms is None:
                        bd_ms = d['bd'] * 1000

                    sw_t, sw_rate = _slide_ls(d['rate_raw'])

                    color = _LS_BLOCKER_COLORS[blocker]
                    sig   = d['significant']
                    alpha = 1.0 if sig else 0.30
                    lw    = 1.8 if sig else 1.0
                    zord  = 3   if sig else 1

                    ax.plot(sw_t, sw_rate,
                            color=color, linewidth=lw, alpha=alpha, zorder=zord,
                            label=BLOCKER_LABEL.get(blocker, blocker))

                    if sig and not np.isnan(d['latency']):
                        ax.axvline(d['latency'], color=color,
                                   linewidth=1.2, linestyle='--', alpha=0.85, zorder=zord)

                if not has_any:
                    ax.set_visible(False)
                    continue

                if mu_ref is not None and sd_ref is not None:
                    ax.axhline(mu_ref, color='#555', ls='--', lw=0.8, alpha=0.5, zorder=0)
                    ax.axhline(mu_ref + K_SD_EXCIT * sd_ref,
                               color=RESP_COLOR['excitatory'], ls=':', lw=0.9, alpha=0.5, zorder=0)
                    ax.axhline(max(MIN_INHIB_THRESHOLD, mu_ref - K_SD_INHIB * sd_ref),
                               color=RESP_COLOR['inhibitory'], ls=':', lw=0.9, alpha=0.5, zorder=0)

                ax.axvline(0, color='k', ls='--', lw=0.9, alpha=0.55, zorder=0)
                if bd_ms is not None and bd_ms > 0:
                    ax.axvspan(0, bd_ms, color='gold', alpha=0.12, zorder=0)
                ax.axvspan(-PRE_TIME_MS, 0, color='lightcyan', alpha=0.12, zorder=0)

                ax.set_xlim(-PRE_TIME_MS, POST_TIME_MS)

                is_shift = (prr, power) in shift_panels

                # Annotate latency shifts per blocker in the panel
                if is_shift:
                    annot_lines = []
                    for blk in BLOCKERS_LATSHIFT:
                        sk = (prr, power, blk)
                        if sk in shift_lookup:
                            lat_nb, lat_blk = shift_lookup[sk]
                            delta = lat_blk - lat_nb
                            sign  = '+' if delta >= 0 else ''
                            blk_short = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}.get(blk, blk)
                            annot_lines.append(f'{blk_short}: {sign}{delta:.0f} ms')
                    if annot_lines:
                        ax.text(0.98, 0.97, '\n'.join(annot_lines),
                                transform=ax.transAxes, ha='right', va='top',
                                fontsize=7, color=_LS_SHIFT_COLOR,
                                bbox=dict(fc='white', ec=_LS_SHIFT_COLOR, lw=0.8, pad=2, alpha=0.85))

                title_str    = f'PRR = {int(prr/1000)} kHz  |  {int(power)} µW'
                title_color  = _LS_SHIFT_COLOR if is_shift else 'black'
                title_suffix = '  ← Δlatency' if is_shift else ''
                ax.set_title(title_str + title_suffix,
                             fontsize=9, fontweight='bold', color=title_color)

                if is_shift:
                    for spine in ax.spines.values():
                        spine.set_edgecolor(_LS_SHIFT_COLOR)
                        spine.set_linewidth(2.0)

                if row_i == len(cell_prr) - 1:
                    ax.set_xlabel('Time (ms)', fontsize=8)
                if col_i == 0:
                    ax.set_ylabel('Firing rate (Hz)', fontsize=8)
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(alpha=0.18)
                ax.tick_params(labelsize=7)

        _blockers_present = [
            b for b in BLOCKER_ORDER
            if any((cluster_id, b, prr, pwr) in zscore_store_all
                   for prr in cell_prr for pwr in cell_power)
        ]
        legend_lines = [
            Line2D([0], [0], color=_LS_BLOCKER_COLORS[b], lw=2,
                   label=BLOCKER_LABEL.get(b, b))
            for b in _blockers_present
        ] + [
            Line2D([0], [0], color='#555', lw=1.2, ls='--', alpha=0.7, label='Baseline (no-blocker)'),
            Line2D([0], [0], color=RESP_COLOR['excitatory'], lw=1.0, ls=':', alpha=0.7, label='Excit. threshold'),
            Line2D([0], [0], color=RESP_COLOR['inhibitory'], lw=1.0, ls=':', alpha=0.7, label='Inhib. threshold'),
        ]
        fig.legend(handles=legend_lines,
                   loc='lower center', ncol=len(legend_lines),
                   bbox_to_anchor=(0.5, -0.06),
                   fontsize=8, framealpha=0.85)

        fig.suptitle(
            f'{cluster_id}  ·  {full_type}  ·  {_ses}  ·  (orange = Δlatency | SW {int(SW_WIN_MS_LS)} ms / {int(SW_STEP_MS_LS)} ms)',
            fontsize=10, fontweight='bold',
        )
        fname = out_dir_ls / f'psth_latshift_{cluster_id}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {len(latshift_cells)} latency-shift PSTH figure(s) → {out_dir_ls}')


## Latency-shift cell counts — per cell type × blocker × PRR × power

Number of cells (ON / OFF / ON-OFF) that show a **same-type latency shift** (≥ `MIN_LATENCY_DIFF_MS`) under ACET or ACET+LAP4, broken down by PRR × laser power.

Subplot grid: **rows = PRR**, **columns = laser power**.  
x-axis = Baden cell type; colour = blocker condition.


In [ ]:
if df.empty or not latshift_cells:
    print('No latency-shift cells — skipping count plot.')
elif not HAS_TYPING:
    print('Cell typing not available — skipping count plot.')
else:
    CT_PLOT_LS      = ['ON', 'OFF', 'ON-OFF']
    BLOCKERS_LS_PLT = ['acet', 'acet lap4']
    BLOCKER_COLORS_LS = BLK_COLOR
    BLOCKER_LABEL_SHORT_LS = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}

    # ── Denominator: total significantly responding cells under no-blocker
    #    per (prr, power, cell_type) — same value for all blocker bars in a panel
    _nb_resp = {}   # (prr, pwr, ct) -> set of cluster_ids
    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        if blk != 'noblocker':
            continue
        if not d['significant']:
            continue
        ct = cid_to_ct.get(cid, 'Unknown')
        if ct not in CT_PLOT_LS:
            continue
        _nb_resp.setdefault((prr, pwr, ct), set()).add(cid)

    denom_nb = {k: len(v) for k, v in _nb_resp.items()}

    # ── Numerator: cells with a latency shift
    _numer_rows = []
    for cid, details in latshift_details.items():
        ct = cid_to_ct.get(cid, 'Unknown')
        if ct not in CT_PLOT_LS:
            continue
        seen = set()
        for prr, pwr, blk, _, _, _ in details:
            if (prr, pwr, blk) in seen:
                continue
            seen.add((prr, pwr, blk))
            _numer_rows.append({'cluster_id': cid, 'cell_type': ct,
                                'prr': prr, 'power': pwr, 'blocker': blk})
    df_numer = pd.DataFrame(_numer_rows).drop_duplicates(
        subset=['cluster_id', 'prr', 'power', 'blocker'])

    _all_prr = sorted({k[0] for k in denom_nb})
    _all_pwr = sorted({k[1] for k in denom_nb})

    if not _all_prr:
        print('No no-blocker responsive cells found.')
    else:
        n_prr, n_pwr = len(_all_prr), len(_all_pwr)

        # x-axis layout (same style as cell 436e9c39)
        within_spacing = 0.3
        between_extra  = 0.1
        box_width      = 0.25

        _x = 0.0
        positions_flat_ls = []
        group_centers_ls  = {}
        group_edges_ls    = {}
        for ct in CT_PLOT_LS:
            xs = []
            for blk in BLOCKERS_LS_PLT:
                positions_flat_ls.append((ct, blk, _x))
                xs.append(_x)
                _x += within_spacing
            group_centers_ls[ct] = np.mean(xs)
            group_edges_ls[ct]   = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
            _x += between_extra

        tick_positions_ls = [xp for _, _, xp in positions_flat_ls]
        tick_labels_ls    = [BLOCKER_LABEL_SHORT_LS[b] for _, b, _ in positions_flat_ls]

        xlim_pad  = box_width * 1.2
        x_span    = (tick_positions_ls[-1] - tick_positions_ls[0]) + 2 * xlim_pad
        fig_width = max(3.5, x_span * 2.0)

        fig, axes = plt.subplots(
            n_prr, n_pwr,
            figsize=(fig_width * n_pwr, 4.5 * n_prr),
            squeeze=False,
        )
        fig.subplots_adjust(bottom=0.30, wspace=0.30, hspace=0.45)

        for row_i, prr in enumerate(_all_prr):
            for col_i, pwr in enumerate(_all_pwr):
                ax = axes[row_i, col_i]

                sub_n = df_numer[(df_numer['prr'] == prr) & (df_numer['power'] == pwr)]
                numer_cnt = (sub_n.groupby(['cell_type', 'blocker'])['cluster_id']
                               .nunique()
                               .reindex(pd.MultiIndex.from_product(
                                   [CT_PLOT_LS, BLOCKERS_LS_PLT],
                                   names=['cell_type', 'blocker']), fill_value=0))

                y_max = 0
                for ct, blk, xp in positions_flat_ls:
                    n   = int(numer_cnt.get((ct, blk), 0))
                    d_  = denom_nb.get((prr, pwr, ct), 0)
                    frac = n / d_ if d_ > 0 else 0.0
                    color = BLOCKER_COLORS_LS[blk]
                    ax.bar(xp, frac, width=box_width, color=color,
                           edgecolor='white', linewidth=0.6, zorder=2)
                    if d_ > 0:
                        ax.text(xp, frac + 0.01, f'{n}/{d_}',
                                ha='center', va='bottom', fontsize=6.5)
                    y_max = max(y_max, frac)

                ax.set_xlim(tick_positions_ls[0] - xlim_pad,
                            tick_positions_ls[-1] + xlim_pad)
                ax.set_ylim(0, min(max(y_max + 0.15, 0.3), 1.05))
                ax.yaxis.set_major_formatter(
                    plt.matplotlib.ticker.PercentFormatter(xmax=1, decimals=0))
                ax.set_xticks(tick_positions_ls)
                ax.set_xticklabels(tick_labels_ls, fontsize=7, rotation=30, ha='right')
                ax.set_ylabel('% no-blocker responsive cells', fontsize=8)
                ax.set_title(f'PRR = {int(prr/1000)} kHz  |  {_pwr_label(pwr)}',
                             fontsize=9, fontweight='bold')
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(axis='y', alpha=0.25)
                ax.tick_params(axis='y', labelsize=7)

                # Group annotations below x-axis
                for ct in CT_PLOT_LS:
                    cx       = group_centers_ls[ct]
                    ex0, ex1 = group_edges_ls[ct]
                    ax.annotate(
                        '', xy=(ex0, 0), xytext=(ex1, 0),
                        xycoords=('data', 'axes fraction'),
                        textcoords=('data', 'axes fraction'),
                        annotation_clip=False,
                        arrowprops=dict(arrowstyle='-', color='#333',
                                        lw=1.0, shrinkA=0, shrinkB=0),
                    )
                    # Show cell-type label with total no-blocker responder count
                    d_on  = denom_nb.get((prr, pwr, ct), 0)
                    ax.annotate(
                        f'{ct}\n(n={d_on})',
                        xy=(cx, 0), xycoords=('data', 'axes fraction'),
                        xytext=(0, -30), textcoords='offset points',
                        ha='center', va='top', fontsize=7.5, fontweight='bold',
                        annotation_clip=False,
                    )
                ax.set_xlabel('Cell type', labelpad=40, fontsize=8)

        legend_patches = [
            mpatches.Patch(color=BLOCKER_COLORS_LS[b],
                           label=BLOCKER_LABEL_SHORT_LS[b])
            for b in BLOCKERS_LS_PLT
        ]
        fig.legend(handles=legend_patches,
                   loc='lower center', ncol=len(BLOCKERS_LS_PLT),
                   bbox_to_anchor=(0.5, 0.01),
                   fontsize=9, framealpha=0.85,
                   title='Blocker condition', title_fontsize=8)

        fig.suptitle(
            f'Fraction of no-blocker responsive cells with same-type latency shift (≥{MIN_LATENCY_DIFF_MS:.0f} ms)'
            f' — per cell type, blocker & PRR × power\n'
            f'(n/d: cells with shift / total no-blocker responders of that type)',
            fontsize=10, fontweight='bold', y=1.02,
        )

        fname = figure_dir / 'summary_latshift_fraction_by_type.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved → {fname}')


## Summary — response outcome proportions vs. no-blocker

For each cell type (ON / OFF / ON-OFF), proportion of no-blocker responsive cells (pooled across all PRR × power) that fall into each outcome category under ACET or ACET+LAP4:

1. **Loss of response** — no significant response under blocker for a condition where no-blocker was significant  
2. **Response reversal** — response type flips (excitatory ↔ inhibitory)  
3. **Latency increase** — same type, latency increases by ≥ `MIN_LATENCY_DIFF_MS`  
4. **Latency decrease** — same type, latency decreases by ≥ `MIN_LATENCY_DIFF_MS`  

A cell is counted once per category if it shows the behaviour in **any** PRR × power condition.  
Denominator = total no-blocker responsive cells of that type (pooled across all PRR × power).


In [ ]:
if df.empty:
    print('No data — skipping summary proportion plot.')
elif not HAS_TYPING:
    print('Cell typing not available — skipping.')
else:
    CT_SUMM      = ['ON', 'OFF', 'ON-OFF']
    BLKS_SUMM    = ['acet', 'acet lap4']
    BLK_COLORS_S = BLK_COLOR
    BLK_LABEL_S  = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}

    CATEGORIES = [
        ('loss',     'Loss of\nresponse',   '#757575'),
        ('reversal', 'Response\nreversal',   '#C0392B'),
        ('lat_pos',  'Latency\nincrease',    '#2980B9'),
        ('lat_neg',  'Latency\ndecrease',    '#8E44AD'),
    ]

    # ── Denominator: unique no-blocker significant cells per cell type
    #    pooled across all (prr, power) conditions
    _nb_pool = {}
    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        if blk != 'noblocker' or not d['significant']:
            continue
        ct = cid_to_ct.get(cid, 'Unknown')
        if ct not in CT_SUMM:
            continue
        _nb_pool.setdefault(ct, set()).add(cid)
    denom_pool = {ct: len(s) for ct, s in _nb_pool.items()}

    # ── Category 1: Loss of response
    #    nb significant → blocker NOT significant (same prr, power)
    _loss_cids = {}
    for (cid, blk, prr, pwr), d in zscore_store_all.items():
        if blk not in BLKS_SUMM or d['significant']:
            continue
        nb_d = zscore_store_all.get((cid, 'noblocker', prr, pwr))
        if nb_d is None or not nb_d['significant']:
            continue
        ct = cid_to_ct.get(cid, 'Unknown')
        if ct not in CT_SUMM:
            continue
        _loss_cids.setdefault((ct, blk), set()).add(cid)

    # ── Category 2: Reversal (from reversal_details)
    _rev_cids = {}
    for cid, details in reversal_details.items():
        ct = cid_to_ct.get(cid, 'Unknown')
        if ct not in CT_SUMM:
            continue
        for prr, pwr, blk in details:
            if blk not in BLKS_SUMM:
                continue
            _rev_cids.setdefault((ct, blk), set()).add(cid)

    # ── Categories 3 & 4: Latency shift direction (from latshift_details)
    _latpos_cids = {}
    _latneg_cids = {}
    for cid, details in latshift_details.items():
        ct = cid_to_ct.get(cid, 'Unknown')
        if ct not in CT_SUMM:
            continue
        for prr, pwr, blk, lat_nb, lat_blk, _ in details:
            if blk not in BLKS_SUMM:
                continue
            if lat_blk > lat_nb:
                _latpos_cids.setdefault((ct, blk), set()).add(cid)
            else:
                _latneg_cids.setdefault((ct, blk), set()).add(cid)

    cat_data = {
        'loss':     _loss_cids,
        'reversal': _rev_cids,
        'lat_pos':  _latpos_cids,
        'lat_neg':  _latneg_cids,
    }

    # ── x-axis layout (2 bars per cell-type group: ACET, ACET+LAP4)
    within_spacing = 0.28
    between_extra  = 0.15
    box_width      = 0.22

    _x = 0.0
    pos_flat = []
    grp_ctr  = {}
    grp_edg  = {}
    for ct in CT_SUMM:
        xs = []
        for blk in BLKS_SUMM:
            pos_flat.append((ct, blk, _x))
            xs.append(_x)
            _x += within_spacing
        grp_ctr[ct] = np.mean(xs)
        grp_edg[ct] = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
        _x += between_extra

    tick_pos = [xp for _, _, xp in pos_flat]
    tick_lbl = [BLK_LABEL_S[b] for _, b, _ in pos_flat]
    xlim_pad = box_width * 1.2
    x_span   = (tick_pos[-1] - tick_pos[0]) + 2 * xlim_pad
    pw       = max(3.2, x_span * 2.0)

    fig, axes = plt.subplots(1, 4, figsize=(pw * 4, 4.8), squeeze=True)
    fig.subplots_adjust(bottom=0.30, wspace=0.38)

    for ax, (cat_key, cat_title, cat_color) in zip(axes, CATEGORIES):
        cids_dict = cat_data[cat_key]

        y_max = 0
        for ct, blk, xp in pos_flat:
            n    = len(cids_dict.get((ct, blk), set()))
            d_   = denom_pool.get(ct, 0)
            frac = n / d_ if d_ > 0 else 0.0
            ax.bar(xp, frac, width=box_width, color=BLK_COLORS_S[blk],
                   edgecolor='white', linewidth=0.6, zorder=2)
            if d_ > 0:
                ax.text(xp, frac + 0.008, f'{n}/{d_}',
                        ha='center', va='bottom', fontsize=6.5)
            y_max = max(y_max, frac)

        ax.set_xlim(tick_pos[0] - xlim_pad, tick_pos[-1] + xlim_pad)
        ax.set_ylim(0, min(max(y_max + 0.15, 0.25), 1.05))
        ax.yaxis.set_major_formatter(
            plt.matplotlib.ticker.PercentFormatter(xmax=1, decimals=0))
        ax.set_xticks(tick_pos)
        ax.set_xticklabels(tick_lbl, fontsize=7, rotation=30, ha='right')
        ax.set_ylabel('% no-blocker responsive cells', fontsize=8)
        ax.set_title(cat_title, fontsize=10, fontweight='bold', color=cat_color)
        ax.spines[['top', 'right']].set_visible(False)
        ax.grid(axis='y', alpha=0.25)
        ax.tick_params(axis='y', labelsize=7)

        for ct in CT_SUMM:
            cx       = grp_ctr[ct]
            ex0, ex1 = grp_edg[ct]
            ax.annotate(
                '', xy=(ex0, 0), xytext=(ex1, 0),
                xycoords=('data', 'axes fraction'),
                textcoords=('data', 'axes fraction'),
                annotation_clip=False,
                arrowprops=dict(arrowstyle='-', color='#333',
                                lw=1.0, shrinkA=0, shrinkB=0),
            )
            d_n = denom_pool.get(ct, 0)
            ax.annotate(
                f'{ct}\n(n={d_n})',
                xy=(cx, 0), xycoords=('data', 'axes fraction'),
                xytext=(0, -30), textcoords='offset points',
                ha='center', va='top', fontsize=7.5, fontweight='bold',
                annotation_clip=False,
            )
        ax.set_xlabel('Cell type', labelpad=40, fontsize=8)

    legend_patches = [
        mpatches.Patch(color=BLK_COLORS_S[b], label=BLK_LABEL_S[b])
        for b in BLKS_SUMM
    ]
    fig.legend(handles=legend_patches,
               loc='lower center', ncol=2,
               bbox_to_anchor=(0.5, 0.01),
               fontsize=9, framealpha=0.85,
               title='Blocker condition', title_fontsize=8)

    fig.suptitle(
        'Response outcome proportions vs. no-blocker — pooled across all PRR × power\n'
        f'Latency threshold: ≥{MIN_LATENCY_DIFF_MS:.0f} ms  |  n/d = cells in category / no-blocker responders',
        fontsize=10, fontweight='bold', y=1.02,
    )

    fname = figure_dir / 'summary_response_outcomes.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {fname}')


## PSTHs — cells with latency decrease under blocker

Cells from `latshift_details` where response latency is **shorter** under ACET or ACET+LAP4 than under no-blocker (Δ < 0, same response type).  
Panels with a latency decrease are highlighted with a **purple** frame and annotated with the Δ per blocker.  
Sliding window: 20 ms / 5 ms step on raw firing rate.


In [ ]:
# Extract cells with at least one latency decrease
_latneg_details = {}   # cid -> [(prr, pwr, blk, lat_nb, lat_blk, resp_type), ...]
for cid, details in latshift_details.items():
    neg = [(prr, pwr, blk, lat_nb, lat_blk, rt)
           for prr, pwr, blk, lat_nb, lat_blk, rt in details
           if lat_blk < lat_nb]
    if neg:
        _latneg_details[cid] = neg

latneg_cells = set(_latneg_details.keys())
print(f'Cells with latency decrease: {len(latneg_cells)}')

if df.empty or not latneg_cells:
    print('No latency-decrease cells — skipping PSTH plots.')
else:
    _LN_COLOR   = '#8E44AD'   # purple — matches summary plot
    _LN_BLOCKER_COLORS = BLK_COLOR

    _ln_win_bins  = int(20.0 / BIN_SIZE_MS)
    _ln_step_bins = int(5.0  / BIN_SIZE_MS)

    def _slide_ln(rate_raw_arr):
        starts  = range(0, len(rate_raw_arr) - _ln_win_bins + 1, _ln_step_bins)
        sw_rate = np.array([rate_raw_arr[i:i + _ln_win_bins].mean() for i in starts])
        sw_t    = np.array([t_centers[i + _ln_win_bins // 2]        for i in starts])
        return sw_t, sw_rate

    out_dir_ln = figure_dir / 'latneg_cell_psths'
    os.makedirs(out_dir_ln, exist_ok=True)

    _ln_keys = [(cid, blk, prr, pwr) for (cid, blk, prr, pwr) in zscore_store_all
                if cid in latneg_cells]

    for cluster_id in tqdm(sorted(latneg_cells), desc='Latency-decrease PSTHs'):
        full_type = cid_to_fulltype.get(cluster_id, 'Unknown')
        _ses = next(
            (d['session'] for (cid, _, _, _), d in zscore_store_all.items()
             if cid == cluster_id), '?'
        )

        cell_prr   = sorted({k[2] for k in _ln_keys if k[0] == cluster_id})
        cell_power = sorted({k[3] for k in _ln_keys if k[0] == cluster_id})
        if not cell_prr or not cell_power:
            continue

        # Panels with a decrease: (prr, power)
        dec_panels = {(prr, pwr) for prr, pwr, _, _, _, _ in _latneg_details[cluster_id]}
        # Lookup: (prr, power, blocker) -> (lat_nb, lat_blk) for decrease entries
        dec_lookup = {
            (prr, pwr, blk): (lat_nb, lat_blk)
            for prr, pwr, blk, lat_nb, lat_blk, _ in _latneg_details[cluster_id]
        }

        fig, axes = plt.subplots(
            len(cell_prr), len(cell_power),
            figsize=(4.5 * len(cell_power), 3.5 * len(cell_prr)),
            squeeze=False,
            constrained_layout=True,
        )

        for row_i, prr in enumerate(cell_prr):
            for col_i, power in enumerate(cell_power):
                ax = axes[row_i, col_i]

                nb_key = (cluster_id, 'noblocker', prr, power)
                mu_ref = sd_ref = bd_ms = None
                if nb_key in zscore_store_all:
                    _d = zscore_store_all[nb_key]
                    mu_ref = _d['mu']
                    sd_ref = _d['sd']
                    bd_ms  = _d['bd'] * 1000

                has_any = False
                for blocker in BLOCKER_ORDER:
                    key = (cluster_id, blocker, prr, power)
                    if key not in zscore_store_all:
                        continue
                    d = zscore_store_all[key]
                    has_any = True

                    if bd_ms is None:
                        bd_ms = d['bd'] * 1000

                    sw_t, sw_rate = _slide_ln(d['rate_raw'])
                    color = _LN_BLOCKER_COLORS[blocker]
                    sig   = d['significant']
                    alpha = 1.0 if sig else 0.30
                    lw    = 1.8 if sig else 1.0
                    zord  = 3   if sig else 1

                    ax.plot(sw_t, sw_rate,
                            color=color, linewidth=lw, alpha=alpha, zorder=zord,
                            label=BLOCKER_LABEL.get(blocker, blocker))

                    if sig and not np.isnan(d['latency']):
                        ax.axvline(d['latency'], color=color,
                                   linewidth=1.2, linestyle='--',
                                   alpha=0.85, zorder=zord)

                if not has_any:
                    ax.set_visible(False)
                    continue

                if mu_ref is not None and sd_ref is not None:
                    ax.axhline(mu_ref, color='#555', ls='--', lw=0.8, alpha=0.5, zorder=0)
                    ax.axhline(mu_ref + K_SD_EXCIT * sd_ref,
                               color=RESP_COLOR['excitatory'], ls=':', lw=0.9, alpha=0.5, zorder=0)
                    ax.axhline(max(MIN_INHIB_THRESHOLD, mu_ref - K_SD_INHIB * sd_ref),
                               color=RESP_COLOR['inhibitory'], ls=':', lw=0.9, alpha=0.5, zorder=0)

                ax.axvline(0, color='k', ls='--', lw=0.9, alpha=0.55, zorder=0)
                if bd_ms is not None and bd_ms > 0:
                    ax.axvspan(0, bd_ms, color='gold', alpha=0.12, zorder=0)
                ax.axvspan(-PRE_TIME_MS, 0, color='lightcyan', alpha=0.12, zorder=0)

                ax.set_xlim(-PRE_TIME_MS, POST_TIME_MS)

                is_dec = (prr, power) in dec_panels

                if is_dec:
                    annot_lines = []
                    for blk in ['acet', 'acet lap4']:
                        sk = (prr, power, blk)
                        if sk in dec_lookup:
                            lat_nb, lat_blk = dec_lookup[sk]
                            delta = lat_blk - lat_nb
                            blk_short = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}.get(blk, blk)
                            annot_lines.append(f'{blk_short}: {delta:.0f} ms')
                    if annot_lines:
                        ax.text(0.98, 0.97, '\n'.join(annot_lines),
                                transform=ax.transAxes, ha='right', va='top',
                                fontsize=7, color=_LN_COLOR,
                                bbox=dict(fc='white', ec=_LN_COLOR, lw=0.8,
                                          pad=2, alpha=0.85))

                title_str    = f'PRR = {int(prr/1000)} kHz  |  {int(power)} µW'
                title_color  = _LN_COLOR if is_dec else 'black'
                title_suffix = '  ← Δlat < 0' if is_dec else ''
                ax.set_title(title_str + title_suffix,
                             fontsize=9, fontweight='bold', color=title_color)

                if is_dec:
                    for spine in ax.spines.values():
                        spine.set_edgecolor(_LN_COLOR)
                        spine.set_linewidth(2.0)

                if row_i == len(cell_prr) - 1:
                    ax.set_xlabel('Time (ms)', fontsize=8)
                if col_i == 0:
                    ax.set_ylabel('Firing rate (Hz)', fontsize=8)
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(alpha=0.18)
                ax.tick_params(labelsize=7)

        _blockers_present = [
            b for b in BLOCKER_ORDER
            if any((cluster_id, b, prr, pwr) in zscore_store_all
                   for prr in cell_prr for pwr in cell_power)
        ]
        legend_lines = [
            Line2D([0], [0], color=_LN_BLOCKER_COLORS[b], lw=2,
                   label=BLOCKER_LABEL.get(b, b))
            for b in _blockers_present
        ] + [
            Line2D([0], [0], color='#555', lw=1.2, ls='--', alpha=0.7,
                   label='Baseline (no-blocker)'),
            Line2D([0], [0], color=RESP_COLOR['excitatory'], lw=1.0, ls=':',
                   alpha=0.7, label='Excit. threshold'),
            Line2D([0], [0], color=RESP_COLOR['inhibitory'], lw=1.0, ls=':',
                   alpha=0.7, label='Inhib. threshold'),
        ]
        fig.legend(handles=legend_lines,
                   loc='lower center', ncol=len(legend_lines),
                   bbox_to_anchor=(0.5, -0.06),
                   fontsize=8, framealpha=0.85)

        fig.suptitle(
            f'{cluster_id}  ·  {full_type}  ·  {_ses}  ·  (purple = latency decrease | SW 20 ms / 5 ms)',
            fontsize=10, fontweight='bold',
        )
        fname = out_dir_ln / f'psth_latneg_{cluster_id}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {len(latneg_cells)} latency-decrease PSTH figure(s) → {out_dir_ln}')


In [ ]:
# ── Detect cells: responsive under no-blocker, unresponsive under ACET and/or ACET+LAP4 ──
BLOCKERS_LOSS = ['acet', 'acet lap4']
loss_details  = {}   # cid -> [(prr, pwr, blk), ...]

for (cid, blk, prr, pwr), d in zscore_store_all.items():
    if blk not in BLOCKERS_LOSS:
        continue
    if d['significant']:           # still responsive — not a loss
        continue
    nb_d = zscore_store_all.get((cid, 'noblocker', prr, pwr))
    if nb_d is None or not nb_d['significant']:
        continue
    loss_details.setdefault(cid, []).append((prr, pwr, blk))

loss_cells = set(loss_details.keys())
print(f'Cells with loss of response: {len(loss_cells)}')

if not loss_cells:
    print('No loss-of-response cells — skipping PSTH plots.')
else:
    _LOSS_COLOR      = '#757575'   # grey — matches summary plot
    _LOSS_BLK_COLORS = BLK_COLOR

    _lo_win_bins  = int(20.0 / BIN_SIZE_MS)
    _lo_step_bins = int(5.0  / BIN_SIZE_MS)

    def _slide_lo(rate_raw_arr):
        starts  = range(0, len(rate_raw_arr) - _lo_win_bins + 1, _lo_step_bins)
        sw_rate = np.array([rate_raw_arr[i:i + _lo_win_bins].mean() for i in starts])
        sw_t    = np.array([t_centers[i + _lo_win_bins // 2]        for i in starts])
        return sw_t, sw_rate

    out_dir_lo = figure_dir / 'loss_cell_psths'
    os.makedirs(out_dir_lo, exist_ok=True)

    _lo_keys = [(cid, blk, prr, pwr) for (cid, blk, prr, pwr) in zscore_store_all
                if cid in loss_cells]

    for cluster_id in tqdm(sorted(loss_cells), desc='Loss-of-response PSTHs'):
        full_type = cid_to_fulltype.get(cluster_id, 'Unknown')
        _ses = next(
            (d['session'] for (cid, _, _, _), d in zscore_store_all.items()
             if cid == cluster_id), '?'
        )

        cell_prr   = sorted({k[2] for k in _lo_keys if k[0] == cluster_id})
        cell_power = sorted({k[3] for k in _lo_keys if k[0] == cluster_id})
        if not cell_prr or not cell_power:
            continue

        # Panels where loss occurs: set of (prr, pwr)
        loss_panels = {(prr, pwr) for prr, pwr, _ in loss_details[cluster_id]}
        # Which blockers caused loss in each panel: (prr, pwr) -> [blk, ...]
        loss_lookup = {}
        for prr, pwr, blk in loss_details[cluster_id]:
            loss_lookup.setdefault((prr, pwr), []).append(blk)

        fig, axes = plt.subplots(
            len(cell_prr), len(cell_power),
            figsize=(4.5 * len(cell_power), 3.5 * len(cell_prr)),
            squeeze=False,
            constrained_layout=True,
        )

        for row_i, prr in enumerate(cell_prr):
            for col_i, power in enumerate(cell_power):
                ax = axes[row_i, col_i]

                nb_key = (cluster_id, 'noblocker', prr, power)
                mu_ref = sd_ref = bd_ms = None
                if nb_key in zscore_store_all:
                    _d    = zscore_store_all[nb_key]
                    mu_ref = _d['mu']
                    sd_ref = _d['sd']
                    bd_ms  = _d['bd'] * 1000

                has_any = False
                for blocker in BLOCKER_ORDER:
                    key = (cluster_id, blocker, prr, power)
                    if key not in zscore_store_all:
                        continue
                    d = zscore_store_all[key]
                    has_any = True

                    if bd_ms is None:
                        bd_ms = d['bd'] * 1000

                    sw_t, sw_rate = _slide_lo(d['rate_raw'])
                    color = _LOSS_BLK_COLORS[blocker]
                    sig   = d['significant']
                    alpha = 1.0 if sig else 0.35
                    lw    = 1.8 if sig else 1.0
                    zord  = 3   if sig else 1

                    ax.plot(sw_t, sw_rate,
                            color=color, linewidth=lw, alpha=alpha, zorder=zord,
                            label=BLOCKER_LABEL.get(blocker, blocker))

                    if sig and not np.isnan(d['latency']):
                        ax.axvline(d['latency'], color=color,
                                   linewidth=1.2, linestyle='--',
                                   alpha=0.85, zorder=zord)

                if not has_any:
                    ax.set_visible(False)
                    continue

                if mu_ref is not None and sd_ref is not None:
                    ax.axhline(mu_ref, color='#555', ls='--', lw=0.8, alpha=0.5, zorder=0)
                    ax.axhline(mu_ref + K_SD_EXCIT * sd_ref,
                               color=RESP_COLOR['excitatory'], ls=':', lw=0.9, alpha=0.5, zorder=0)
                    ax.axhline(max(MIN_INHIB_THRESHOLD, mu_ref - K_SD_INHIB * sd_ref),
                               color=RESP_COLOR['inhibitory'], ls=':', lw=0.9, alpha=0.5, zorder=0)

                ax.axvline(0, color='k', ls='--', lw=0.9, alpha=0.55, zorder=0)
                if bd_ms is not None and bd_ms > 0:
                    ax.axvspan(0, bd_ms, color='gold', alpha=0.12, zorder=0)
                ax.axvspan(-PRE_TIME_MS, 0, color='lightcyan', alpha=0.12, zorder=0)
                ax.set_xlim(-PRE_TIME_MS, POST_TIME_MS)

                is_loss = (prr, power) in loss_panels

                if is_loss:
                    blk_short = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}
                    lost_in   = ', '.join(blk_short.get(b, b)
                                         for b in loss_lookup[(prr, power)])
                    ax.text(0.98, 0.97, f'Loss: {lost_in}',
                            transform=ax.transAxes, ha='right', va='top',
                            fontsize=7, color=_LOSS_COLOR,
                            bbox=dict(fc='white', ec=_LOSS_COLOR, lw=0.8,
                                      pad=2, alpha=0.85))

                title_str    = f'PRR = {int(prr/1000)} kHz  |  {_pwr_label(power)}'
                title_color  = _LOSS_COLOR if is_loss else 'black'
                title_suffix = '  ← loss' if is_loss else ''
                ax.set_title(title_str + title_suffix,
                             fontsize=9, fontweight='bold', color=title_color)

                if is_loss:
                    for spine in ax.spines.values():
                        spine.set_edgecolor(_LOSS_COLOR)
                        spine.set_linewidth(2.0)

                if row_i == len(cell_prr) - 1:
                    ax.set_xlabel('Time (ms)', fontsize=8)
                if col_i == 0:
                    ax.set_ylabel('Firing rate (Hz)', fontsize=8)
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(alpha=0.18)
                ax.tick_params(labelsize=7)

        _blockers_present = [
            b for b in BLOCKER_ORDER
            if any((cluster_id, b, prr, pwr) in zscore_store_all
                   for prr in cell_prr for pwr in cell_power)
        ]
        legend_lines = [
            Line2D([0], [0], color=_LOSS_BLK_COLORS[b], lw=2,
                   label=BLOCKER_LABEL.get(b, b))
            for b in _blockers_present
        ] + [
            Line2D([0], [0], color='#555', lw=1.2, ls='--', alpha=0.7,
                   label='Baseline (no-blocker)'),
            Line2D([0], [0], color=RESP_COLOR['excitatory'], lw=1.0, ls=':',
                   alpha=0.7, label='Excit. threshold'),
            Line2D([0], [0], color=RESP_COLOR['inhibitory'], lw=1.0, ls=':',
                   alpha=0.7, label='Inhib. threshold'),
        ]
        fig.legend(handles=legend_lines,
                   loc='lower center', ncol=len(legend_lines),
                   bbox_to_anchor=(0.5, -0.06),
                   fontsize=8, framealpha=0.85)

        fig.suptitle(
            f'{cluster_id}  ·  {full_type}  ·  {_ses}  ·  (grey = loss of response | SW 20 ms / 5 ms)',
            fontsize=10, fontweight='bold',
        )
        fname = out_dir_lo / f'psth_loss_{cluster_id}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f'Saved {len(loss_cells)} loss-of-response PSTH figure(s) → {out_dir_lo}')


In [ ]:
# ── Collect Δlatency values from latshift_details, split by direction
CT_LAT   = ['ON', 'OFF', 'ON-OFF']
BLKS_LAT = ['acet', 'acet lap4']
BLK_LABEL_LAT = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}
DIRS = ['positive', 'negative']

# Denominator: unique no-blocker significant cells per cell type, all prr × power pooled
_nb_pool_lat = {}
for (cid, blk, prr, pwr), d in zscore_store_all.items():
    if blk != 'noblocker' or not d['significant']:
        continue
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_LAT:
        continue
    _nb_pool_lat.setdefault(ct, set()).add(cid)
denom_pool_lat = {ct: len(s) for ct, s in _nb_pool_lat.items()}

# delta_store[(direction, ct, blk)] = list of Δlatency values (ms)
delta_store = {(d, ct, blk): [] for d in DIRS for ct in CT_LAT for blk in BLKS_LAT}
# Track unique cells per (direction, ct, blk) for percentage
ncells_store = {(d, ct, blk): set() for d in DIRS for ct in CT_LAT for blk in BLKS_LAT}

for cid, details in latshift_details.items():
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_LAT:
        continue
    for prr, pwr, blk, lat_nb, lat_blk, _ in details:
        if blk not in BLKS_LAT:
            continue
        delta = lat_blk - lat_nb
        direction = 'positive' if delta > 0 else 'negative'
        delta_store[(direction, ct, blk)].append(delta)
        ncells_store[(direction, ct, blk)].add(cid)

# ── Print summary table
print(f'{"Direction":<12} {"Cell type":<10} {"Blocker":<14} {"n":>4}  {"%NB":>6}  {"mean Δ (ms)":>12}  {"SD (ms)":>8}  {"SEM (ms)":>8}')
print('-' * 78)
for direction in DIRS:
    for ct in CT_LAT:
        for blk in BLKS_LAT:
            vals = delta_store[(direction, ct, blk)]
            n    = len(ncells_store[(direction, ct, blk)])
            if len(vals) == 0:
                continue
            denom = denom_pool_lat.get(ct, 0)
            pct   = 100 * n / denom if denom > 0 else 0.0
            mean_d = np.mean(vals)
            sd_d   = np.std(vals, ddof=1) if len(vals) > 1 else 0.0
            sem_d  = sd_d / np.sqrt(len(vals))
            sign   = '+' if mean_d > 0 else ''
            print(f'{direction:<12} {ct:<10} {BLK_LABEL_LAT[blk]:<14} {n:>4}  {pct:>5.1f}%  '
                  f'{sign}{mean_d:>10.1f}   {sd_d:>8.1f}   {sem_d:>8.1f}')
    print()

# ── Bar plot: mean Δlatency ± SEM, label = % of no-blocker responders
BLK_COLORS_LAT = BLK_COLOR
DIR_LABEL = {'positive': 'Δlat > 0 (increase)', 'negative': 'Δlat < 0 (decrease)'}

within_spacing = 0.28
between_extra  = 0.15
box_width      = 0.22

_x = 0.0
pos_flat_lat = []
grp_ctr_lat  = {}
grp_edg_lat  = {}
for ct in CT_LAT:
    xs = []
    for blk in BLKS_LAT:
        pos_flat_lat.append((ct, blk, _x))
        xs.append(_x)
        _x += within_spacing
    grp_ctr_lat[ct] = np.mean(xs)
    grp_edg_lat[ct] = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
    _x += between_extra

tick_pos_lat = [xp for _, _, xp in pos_flat_lat]
tick_lbl_lat = [BLK_LABEL_LAT[b] for _, b, _ in pos_flat_lat]
xlim_pad = box_width * 1.2
x_span   = (tick_pos_lat[-1] - tick_pos_lat[0]) + 2 * xlim_pad
pw       = max(3.5, x_span * 2.0)

fig, axes = plt.subplots(1, 2, figsize=(pw * 2, 4.5), squeeze=True)
fig.subplots_adjust(bottom=0.28, wspace=0.38)

for ax, direction in zip(axes, DIRS):
    y_min, y_max = 0.0, 0.0
    for ct, blk, xp in pos_flat_lat:
        vals  = delta_store[(direction, ct, blk)]
        n_cid = len(ncells_store[(direction, ct, blk)])
        if len(vals) == 0:
            continue
        mean_d = np.mean(vals)
        sem_d  = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
        denom  = denom_pool_lat.get(ct, 0)
        pct    = 100 * n_cid / denom if denom > 0 else 0.0
        color  = BLK_COLORS_LAT[blk]
        ax.bar(xp, mean_d, width=box_width, color=color,
               edgecolor='white', linewidth=0.6, zorder=2)
        ax.errorbar(xp, mean_d, yerr=sem_d, fmt='none',
                    color='#333', capsize=3, linewidth=1.2, zorder=3)
        offset = (sem_d + 1.0) * np.sign(mean_d)
        ax.text(xp, mean_d + offset, f'{pct:.0f}%',
                ha='center',
                va='bottom' if mean_d >= 0 else 'top',
                fontsize=6.5)
        y_min = min(y_min, mean_d - sem_d)
        y_max = max(y_max, mean_d + sem_d)

    ax.axhline(0, color='#333', lw=0.8, ls='-', zorder=1)
    pad = max(abs(y_min), abs(y_max)) * 0.25 + 3
    ax.set_ylim(y_min - pad, y_max + pad)
    ax.set_xlim(tick_pos_lat[0] - xlim_pad, tick_pos_lat[-1] + xlim_pad)
    ax.set_xticks(tick_pos_lat)
    ax.set_xticklabels(tick_lbl_lat, fontsize=7, rotation=30, ha='right')
    ax.set_ylabel('Mean Δ latency (ms)  [blocker − no blocker]', fontsize=8)
    ax.set_title(DIR_LABEL[direction], fontsize=10, fontweight='bold',
                 color='#2980B9' if direction == 'positive' else '#8E44AD')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.25)
    ax.tick_params(axis='y', labelsize=7)

    for ct in CT_LAT:
        cx       = grp_ctr_lat[ct]
        ex0, ex1 = grp_edg_lat[ct]
        ax.annotate('', xy=(ex0, 0), xytext=(ex1, 0),
                    xycoords=('data', 'axes fraction'),
                    textcoords=('data', 'axes fraction'),
                    annotation_clip=False,
                    arrowprops=dict(arrowstyle='-', color='#333',
                                    lw=1.0, shrinkA=0, shrinkB=0))
        d_n = denom_pool_lat.get(ct, 0)
        ax.annotate(f'{ct}\n(n={d_n})',
                    xy=(cx, 0), xycoords=('data', 'axes fraction'),
                    xytext=(0, -28), textcoords='offset points',
                    ha='center', va='top', fontsize=7.5, fontweight='bold',
                    annotation_clip=False)
    ax.set_xlabel('Cell type', labelpad=36, fontsize=8)

legend_patches = [
    mpatches.Patch(color=BLK_COLORS_LAT[b], label=BLK_LABEL_LAT[b])
    for b in BLKS_LAT
]
fig.legend(handles=legend_patches,
           loc='lower center', ncol=2,
           bbox_to_anchor=(0.5, 0.01),
           fontsize=9, framealpha=0.85,
           title='Blocker condition', title_fontsize=8)
fig.suptitle(
    f'Mean latency shift (blocker − no blocker) ± SEM  |  label = % of no-blocker responders\n'
    f'Threshold: ≥{MIN_LATENCY_DIFF_MS:.0f} ms  |  pooled across all PRR × power',
    fontsize=10, fontweight='bold', y=1.02,
)
fname = figure_dir / 'summary_latency_shift_mean.png'
fig.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {fname}')


In [ ]:
# Mean latency shift per PRR × power — positive and negative shifts
# Subplot grid: rows = PRR, cols = power
# Each panel: bars per (cell_type, blocker), bars extend up (pos) or down (neg)
# Bar label = % of no-blocker responders for that (prr, power, cell_type)

CT_LAT2   = ['ON', 'OFF', 'ON-OFF']
BLKS_LAT2 = ['acet', 'acet lap4']
BLK_LABEL_LAT2  = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}
BLK_COLORS_LAT2 = BLK_COLOR

# ── Denominator: no-blocker significant cells per (prr, power, cell_type)
_nb_pool2 = {}
for (cid, blk, prr, pwr), d in zscore_store_all.items():
    if blk != 'noblocker' or not d['significant']:
        continue
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_LAT2:
        continue
    _nb_pool2.setdefault((prr, pwr, ct), set()).add(cid)
denom_pwr = {k: len(v) for k, v in _nb_pool2.items()}

# ── Collect Δlatency per (direction, ct, blk, prr, pwr)
_delta2 = {}    # (direction, ct, blk, prr, pwr) -> list of Δ values
_ncid2  = {}    # (direction, ct, blk, prr, pwr) -> set of cids

for cid, details in latshift_details.items():
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_LAT2:
        continue
    for prr, pwr, blk, lat_nb, lat_blk, _ in details:
        if blk not in BLKS_LAT2:
            continue
        delta     = lat_blk - lat_nb
        direction = 'positive' if delta > 0 else 'negative'
        key = (direction, ct, blk, prr, pwr)
        _delta2.setdefault(key, []).append(delta)
        _ncid2.setdefault(key, set()).add(cid)

all_prr2 = sorted({k[3] for k in _delta2})
all_pwr2 = sorted({k[4] for k in _delta2})

if not all_prr2:
    print('No latency-shift data — skipping PRR × power plot.')
else:
    n_prr2, n_pwr2 = len(all_prr2), len(all_pwr2)

    # x-axis layout (same style as other cells)
    within_spacing = 0.28
    between_extra  = 0.15
    box_width      = 0.22

    _x = 0.0
    pos_flat2 = []
    grp_ctr2  = {}
    grp_edg2  = {}
    for ct in CT_LAT2:
        xs = []
        for blk in BLKS_LAT2:
            pos_flat2.append((ct, blk, _x))
            xs.append(_x)
            _x += within_spacing
        grp_ctr2[ct] = np.mean(xs)
        grp_edg2[ct] = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
        _x += between_extra

    tick_pos2 = [xp for _, _, xp in pos_flat2]
    tick_lbl2 = [BLK_LABEL_LAT2[b] for _, b, _ in pos_flat2]
    xlim_pad  = box_width * 1.2
    x_span    = (tick_pos2[-1] - tick_pos2[0]) + 2 * xlim_pad
    pw        = max(3.5, x_span * 2.0)

    # 2 figures: one per direction
    for direction, dir_color, dir_label, dir_fname in [
        ('positive', '#2980B9', 'Latency increase (Δ > 0)', 'latshift_by_prr_pwr_increase.png'),
        ('negative', '#8E44AD', 'Latency decrease (Δ < 0)', 'latshift_by_prr_pwr_decrease.png'),
    ]:
        fig, axes = plt.subplots(
            n_prr2, n_pwr2,
            figsize=(pw * n_pwr2, 4.5 * n_prr2),
            squeeze=False,
        )
        fig.subplots_adjust(bottom=0.28, wspace=0.35, hspace=0.45)

        for row_i, prr in enumerate(all_prr2):
            for col_i, pwr in enumerate(all_pwr2):
                ax = axes[row_i, col_i]

                y_min, y_max = 0.0, 0.0
                has_data = False

                for ct, blk, xp in pos_flat2:
                    key   = (direction, ct, blk, prr, pwr)
                    vals  = _delta2.get(key, [])
                    n_cid = len(_ncid2.get(key, set()))
                    denom = denom_pwr.get((prr, pwr, ct), 0)
                    if len(vals) == 0 or denom == 0:
                        continue
                    has_data = True
                    mean_d = np.mean(vals)
                    sem_d  = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0.0
                    pct    = 100 * n_cid / denom

                    ax.bar(xp, mean_d, width=box_width,
                           color=BLK_COLORS_LAT2[blk],
                           edgecolor='white', linewidth=0.6, zorder=2)
                    ax.errorbar(xp, mean_d, yerr=sem_d, fmt='none',
                                color='#333', capsize=3, linewidth=1.2, zorder=3)
                    offset = (sem_d + 1.0) * np.sign(mean_d)
                    ax.text(xp, mean_d + offset, f'{pct:.0f}%',
                            ha='center',
                            va='bottom' if mean_d >= 0 else 'top',
                            fontsize=6.5)
                    y_min = min(y_min, mean_d - sem_d)
                    y_max = max(y_max, mean_d + sem_d)

                ax.axhline(0, color='#333', lw=0.8, ls='-', zorder=1)
                pad = max(abs(y_min), abs(y_max)) * 0.25 + 3
                ax.set_ylim(y_min - pad if has_data else -5,
                            y_max + pad if has_data else 5)
                ax.set_xlim(tick_pos2[0] - xlim_pad, tick_pos2[-1] + xlim_pad)
                ax.set_xticks(tick_pos2)
                ax.set_xticklabels(tick_lbl2, fontsize=7, rotation=30, ha='right')
                ax.set_ylabel('Mean Δ latency (ms)', fontsize=8)
                ax.set_title(f'PRR = {int(prr/1000)} kHz  |  {_pwr_label(pwr)}',
                             fontsize=9, fontweight='bold')
                ax.spines[['top', 'right']].set_visible(False)
                ax.grid(axis='y', alpha=0.25)
                ax.tick_params(axis='y', labelsize=7)

                for ct in CT_LAT2:
                    cx       = grp_ctr2[ct]
                    ex0, ex1 = grp_edg2[ct]
                    ax.annotate('', xy=(ex0, 0), xytext=(ex1, 0),
                                xycoords=('data', 'axes fraction'),
                                textcoords=('data', 'axes fraction'),
                                annotation_clip=False,
                                arrowprops=dict(arrowstyle='-', color='#333',
                                                lw=1.0, shrinkA=0, shrinkB=0))
                    d_n = denom_pwr.get((prr, pwr, ct), 0)
                    ax.annotate(f'{ct}\n(n={d_n})',
                                xy=(cx, 0), xycoords=('data', 'axes fraction'),
                                xytext=(0, -28), textcoords='offset points',
                                ha='center', va='top', fontsize=7.5, fontweight='bold',
                                annotation_clip=False)
                ax.set_xlabel('Cell type', labelpad=36, fontsize=8)

        legend_patches = [
            mpatches.Patch(color=BLK_COLORS_LAT2[b], label=BLK_LABEL_LAT2[b])
            for b in BLKS_LAT2
        ]
        fig.legend(handles=legend_patches,
                   loc='lower center', ncol=2,
                   bbox_to_anchor=(0.5, 0.01),
                   fontsize=9, framealpha=0.85,
                   title='Blocker condition', title_fontsize=8)
        fig.suptitle(
            f'{dir_label} — mean Δ latency (blocker − no blocker) ± SEM\n'
            f'Label = % of no-blocker responders for that PRR × power × cell type  '
            f'(threshold ≥{MIN_LATENCY_DIFF_MS:.0f} ms)',
            fontsize=10, fontweight='bold', color=dir_color, y=1.02,
        )
        fname = figure_dir / dir_fname
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved → {fname}')


In [ ]:
# Response reversal proportion — per PRR × power × blocker × cell type
# Subplot grid: rows = PRR, cols = power
# Bar height = % of no-blocker responders (for that prr, power, cell_type) showing reversal
# Denominator = no-blocker significant cells per (prr, power, cell_type)

CT_REV   = ['ON', 'OFF', 'ON-OFF']
BLKS_REV = ['acet', 'acet lap4']
BLK_COLORS_REV = BLK_COLOR
BLK_LABEL_REV  = {'acet': 'ACET', 'acet lap4': 'ACET+LAP4'}

# ── Denominator: no-blocker significant cells per (prr, power, cell_type)
_nb_pool_rev = {}
for (cid, blk, prr, pwr), d in zscore_store_all.items():
    if blk != 'noblocker' or not d['significant']:
        continue
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_REV:
        continue
    _nb_pool_rev.setdefault((prr, pwr, ct), set()).add(cid)
denom_rev = {k: len(v) for k, v in _nb_pool_rev.items()}

# ── Numerator: reversal cells per (prr, power, blocker, cell_type)
_rev_cids2 = {}   # (prr, pwr, blk, ct) -> set of cids
for cid, details in reversal_details.items():
    ct = cid_to_ct.get(cid, 'Unknown')
    if ct not in CT_REV:
        continue
    for prr, pwr, blk in details:
        if blk not in BLKS_REV:
            continue
        _rev_cids2.setdefault((prr, pwr, blk, ct), set()).add(cid)

all_prr_rev = sorted({k[0] for k in denom_rev})
all_pwr_rev = sorted({k[1] for k in denom_rev})

if not all_prr_rev:
    print('No data — skipping reversal PRR × power plot.')
else:
    n_prr_rev, n_pwr_rev = len(all_prr_rev), len(all_pwr_rev)

    # x-axis layout
    within_spacing = 0.28
    between_extra  = 0.15
    box_width      = 0.22

    _x = 0.0
    pos_flat_rev = []
    grp_ctr_rev  = {}
    grp_edg_rev  = {}
    for ct in CT_REV:
        xs = []
        for blk in BLKS_REV:
            pos_flat_rev.append((ct, blk, _x))
            xs.append(_x)
            _x += within_spacing
        grp_ctr_rev[ct] = np.mean(xs)
        grp_edg_rev[ct] = (xs[0] - box_width / 2, xs[-1] + box_width / 2)
        _x += between_extra

    tick_pos_rev = [xp for _, _, xp in pos_flat_rev]
    tick_lbl_rev = [BLK_LABEL_REV[b] for _, b, _ in pos_flat_rev]
    xlim_pad = box_width * 1.2
    x_span   = (tick_pos_rev[-1] - tick_pos_rev[0]) + 2 * xlim_pad
    pw       = max(3.5, x_span * 2.0)

    fig, axes = plt.subplots(
        n_prr_rev, n_pwr_rev,
        figsize=(pw * n_pwr_rev, 4.5 * n_prr_rev),
        squeeze=False,
    )
    fig.subplots_adjust(bottom=0.28, wspace=0.35, hspace=0.45)

    for row_i, prr in enumerate(all_prr_rev):
        for col_i, pwr in enumerate(all_pwr_rev):
            ax = axes[row_i, col_i]

            y_max = 0
            for ct, blk, xp in pos_flat_rev:
                n     = len(_rev_cids2.get((prr, pwr, blk, ct), set()))
                denom = denom_rev.get((prr, pwr, ct), 0)
                frac  = n / denom if denom > 0 else 0.0
                color = BLK_COLORS_REV[blk]
                ax.bar(xp, frac, width=box_width, color=color,
                       edgecolor='white', linewidth=0.6, zorder=2)
                if denom > 0:
                    ax.text(xp, frac + 0.008, f'{100*frac:.0f}%',
                            ha='center', va='bottom', fontsize=6.5)
                y_max = max(y_max, frac)

            ax.set_xlim(tick_pos_rev[0] - xlim_pad, tick_pos_rev[-1] + xlim_pad)
            ax.set_ylim(0, min(max(y_max + 0.15, 0.20), 1.05))
            ax.yaxis.set_major_formatter(
                plt.matplotlib.ticker.PercentFormatter(xmax=1, decimals=0))
            ax.set_xticks(tick_pos_rev)
            ax.set_xticklabels(tick_lbl_rev, fontsize=7, rotation=30, ha='right')
            ax.set_ylabel('% no-blocker responsive cells', fontsize=8)
            ax.set_title(f'PRR = {int(prr/1000)} kHz  |  {_pwr_label(pwr)}',
                         fontsize=9, fontweight='bold')
            ax.spines[['top', 'right']].set_visible(False)
            ax.grid(axis='y', alpha=0.25)
            ax.tick_params(axis='y', labelsize=7)

            for ct in CT_REV:
                cx       = grp_ctr_rev[ct]
                ex0, ex1 = grp_edg_rev[ct]
                ax.annotate('', xy=(ex0, 0), xytext=(ex1, 0),
                            xycoords=('data', 'axes fraction'),
                            textcoords=('data', 'axes fraction'),
                            annotation_clip=False,
                            arrowprops=dict(arrowstyle='-', color='#333',
                                            lw=1.0, shrinkA=0, shrinkB=0))
                d_n = denom_rev.get((prr, pwr, ct), 0)
                ax.annotate(f'{ct}\n(n={d_n})',
                            xy=(cx, 0), xycoords=('data', 'axes fraction'),
                            xytext=(0, -28), textcoords='offset points',
                            ha='center', va='top', fontsize=7.5, fontweight='bold',
                            annotation_clip=False)
            ax.set_xlabel('Cell type', labelpad=36, fontsize=8)

    legend_patches = [
        mpatches.Patch(color=BLK_COLORS_REV[b], label=BLK_LABEL_REV[b])
        for b in BLKS_REV
    ]
    fig.legend(handles=legend_patches,
               loc='lower center', ncol=2,
               bbox_to_anchor=(0.5, 0.01),
               fontsize=9, framealpha=0.85,
               title='Blocker condition', title_fontsize=8)
    fig.suptitle(
        'Response reversal (excitatory ↔ inhibitory) — % of no-blocker responsive cells\n'
        'per cell type × blocker × PRR × power',
        fontsize=10, fontweight='bold', color='#C0392B', y=1.02,
    )
    fname = figure_dir / 'summary_reversal_by_prr_pwr.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {fname}')


## A. Population response matrix & B. Response trajectory (No blocker → ACET+LAP4)

**A. Heatmap** — rows = individual cells sorted by no-blocker response class; columns = No blocker / ACET / ACET+LAP4.  
Colour = (mean post-stim rate − no-blocker baseline) / no-blocker baseline SD, clipped to ±5.  

**B. Scatter** — each point = one cell; x = normalised Δrate under no-blocker; y = normalised Δrate under ACET+LAP4.  
Reference lines: `y = x` (no change), `y = −x` (perfect reversal), `y = 0` (loss of response).

One figure per PRR × power condition, saved to `population_response/`.


In [ ]:
import matplotlib.colors as _mcolors

_RESP_COLORS_AB = {'excitatory': '#E63946', 'inhibitory': '#457B9D', 'none': '#CCCCCC'}
_CLASS_RANK_AB  = {'excitatory': 0, 'inhibitory': 1, 'none': 2}
_BLK_KEYS_AB    = ['noblocker', 'acet', 'acet lap4']   # internal keys
_BLK_NAMES_AB   = ['No blocker', 'ACET', 'ACET+LAP4']  # display names
_COL_Z          = ['z_nb', 'z_ac', 'z_al']
_COL_RC         = ['rc_nb', 'rc_ac', 'rc_al']

_prr_ab = sorted({k[2] for k in zscore_store_all})
_pwr_ab = sorted({k[3] for k in zscore_store_all})

out_dir_ab = figure_dir / 'population_response'
os.makedirs(out_dir_ab, exist_ok=True)

for _prr in _prr_ab:
    for _pwr in _pwr_ab:
        # ── Build per-cell table ──────────────────────────────────────────────
        _cids = {k[0] for k in zscore_store_all if k[2] == _prr and k[3] == _pwr}
        _rows = []
        for cid in _cids:
            nb_d = zscore_store_all.get((cid, 'noblocker', _prr, _pwr))
            if nb_d is None:
                continue
            mu_ref = nb_d['mu']
            sd_ref = max(nb_d['sd'], 0.1)
            row = {'cid': cid}
            for col_z, col_rc, blk in zip(_COL_Z, _COL_RC, _BLK_KEYS_AB):
                d = zscore_store_all.get((cid, blk, _prr, _pwr))
                if d is not None:
                    row[col_z]  = (d['mean_post_rate'] - mu_ref) / sd_ref
                    row[col_rc] = d['response_type'] if d['significant'] else 'none'
                else:
                    row[col_z]  = np.nan
                    row[col_rc] = 'none'
            _rows.append(row)

        if not _rows:
            continue

        df_ab = pd.DataFrame(_rows)
        df_ab['_rank'] = df_ab['rc_nb'].map(_CLASS_RANK_AB).fillna(2)
        df_ab = df_ab.sort_values(['_rank', 'z_nb'], ascending=[True, False]).reset_index(drop=True)

        n_cells = len(df_ab)
        Z_mat   = np.clip(df_ab[_COL_Z].values.astype(float), -5, 5)
        row_colors = [_RESP_COLORS_AB.get(r, '#CCCCCC') for r in df_ab['rc_nb']]

        # ── Figure layout ─────────────────────────────────────────────────────
        fig_h = max(6.0, n_cells * max(0.06, min(0.22, 8.0 / n_cells)) + 2.5)
        fig = plt.figure(figsize=(15, fig_h))
        gs  = fig.add_gridspec(
            1, 3,
            width_ratios=[0.04, 0.30, 0.62],
            wspace=0.04,
            left=0.04, right=0.96,
            bottom=0.11, top=0.90,
        )
        ax_lbl  = fig.add_subplot(gs[0])
        ax_heat = fig.add_subplot(gs[1])
        ax_scat = fig.add_subplot(gs[2])

        # ── A. Heatmap ────────────────────────────────────────────────────────
        # Row-colour sidebar
        color_rgb = np.array([_mcolors.to_rgb(c) for c in row_colors]).reshape(n_cells, 1, 3)
        ax_lbl.imshow(color_rgb, aspect='auto', interpolation='none')
        ax_lbl.set_xticks([]); ax_lbl.set_yticks([])
        ax_lbl.set_xlabel('Class
(No blk)', fontsize=6.5)

        im = ax_heat.imshow(Z_mat, aspect='auto', cmap='RdBu_r',
                            vmin=-5, vmax=5, interpolation='none')
        ax_heat.set_xticks([0, 1, 2])
        ax_heat.set_xticklabels(_BLK_NAMES_AB, fontsize=8)
        ax_heat.set_yticks([])
        ax_heat.set_ylabel(f'Individual cells  (n = {n_cells})', fontsize=8)
        ax_heat.set_title('A   Population response matrix',
                          fontsize=9, fontweight='bold', loc='left')

        # Class boundary lines + right-side labels
        prev_rc = df_ab['rc_nb'].iloc[0]
        class_spans = {}
        for i, rc in enumerate(df_ab['rc_nb']):
            if rc not in class_spans:
                class_spans[rc] = [i, i]
            else:
                class_spans[rc][1] = i
            if rc != prev_rc:
                ax_heat.axhline(i - 0.5, color='k', lw=0.8)
                ax_lbl.axhline( i - 0.5, color='k', lw=0.8)
                prev_rc = rc

        for rc_class, (s, e) in class_spans.items():
            mid   = (s + e) / 2
            label = {'excitatory': 'Excitatory', 'inhibitory': 'Inhibitory',
                     'none': 'No response'}.get(rc_class, rc_class)
            ax_heat.text(2.55, mid, label,
                         va='center', ha='left', fontsize=7,
                         color=_RESP_COLORS_AB.get(rc_class, '#333'),
                         fontweight='bold', clip_on=False)

        cbar = plt.colorbar(im, ax=ax_heat, shrink=0.45, pad=0.18, aspect=25)
        cbar.set_label('Δrate / baseline SD', fontsize=7)
        cbar.ax.tick_params(labelsize=7)

        # ── B. Scatter ────────────────────────────────────────────────────────
        df_scat = df_ab.dropna(subset=['z_nb', 'z_al'])
        z_lim   = max(df_scat[['z_nb', 'z_al']].abs().max().max() * 1.1, 3.5)

        for rc_class in ['none', 'inhibitory', 'excitatory']:
            sub = df_scat[df_scat['rc_nb'] == rc_class]
            if sub.empty:
                continue
            ax_scat.scatter(
                sub['z_nb'], sub['z_al'],
                c=_RESP_COLORS_AB.get(rc_class, '#CCC'),
                s=22, alpha=0.75, linewidths=0.3, edgecolors='white', zorder=3,
                label={'excitatory': 'Excitatory (no-blk)',
                       'inhibitory': 'Inhibitory (no-blk)',
                       'none':       'No response (no-blk)'}.get(rc_class),
            )

        _ls = np.array([-z_lim, z_lim])
        ax_scat.plot(_ls,  _ls, color='#444',    lw=1.3, ls='--', alpha=0.65,
                     label='y = x  (no change)')
        ax_scat.plot(_ls, -_ls, color='#C0392B', lw=1.3, ls=':',  alpha=0.85,
                     label='y = −x  (reversal)')
        ax_scat.axhline(0, color='#8E44AD', lw=1.1, ls='-.', alpha=0.65,
                        label='y = 0  (loss of response)')
        ax_scat.axvline(0, color='#888', lw=0.6, alpha=0.3)

        ax_scat.set_xlim(-z_lim, z_lim)
        ax_scat.set_ylim(-z_lim, z_lim)
        ax_scat.set_aspect('equal', adjustable='box')
        ax_scat.set_xlabel('Δrate / SD — No blocker', fontsize=9)
        ax_scat.set_ylabel('Δrate / SD — ACET + LAP4', fontsize=9)
        ax_scat.set_title('B   Response trajectory  (No blocker → ACET+LAP4)',
                          fontsize=9, fontweight='bold', loc='left')
        ax_scat.spines[['top', 'right']].set_visible(False)
        ax_scat.legend(fontsize=7.5, loc='upper left', framealpha=0.9)
        ax_scat.tick_params(labelsize=7)
        ax_scat.grid(alpha=0.15)

        fig.suptitle(
            f'PRR = {int(_prr/1000)} kHz  |  {_pwr_label(_pwr)}\n'
            'Colour scale: (post-stim rate − no-blocker baseline) / no-blocker baseline SD',
            fontsize=10, fontweight='bold',
        )
        fname = out_dir_ab / f'population_response_prr{int(_prr)}_pwr{int(_pwr)}.png'
        fig.savefig(fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'Saved: {fname.name}')


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
TARGET_CID    = 'uid_260423_108'
TARGET_PRR    = 5000
TARGET_PWR    = 5000
N_BOOT_UID    = 1000
N_TRIALS_UID  = 50    # ← set to actual number of trials per stimulus condition
CI_BOUNDS_UID = (2.5, 97.5)

_SW_WIN_B  = int(20.0 / BIN_SIZE_MS)   # 40 bins
_SW_STEP_B = int(5.0  / BIN_SIZE_MS)   # 10 bins
_BIN_S_UID = BIN_SIZE_MS / 1000.0

BLOCKER_ORDER_UID = ['noblocker', 'acet', 'acet lap4', 'washout']
BLOCKER_COLOR_UID = BLK_COLOR
BLOCKER_LABEL_UID = {
    'noblocker': 'No blocker', 'acet': 'ACET',
    'acet lap4': 'ACET+LAP4',  'washout': 'Washout',
}

def _uid_slide(arr):
    starts = range(0, len(arr) - _SW_WIN_B + 1, _SW_STEP_B)
    t = np.array([t_centers[s + _SW_WIN_B // 2] for s in starts])
    r = np.array([arr[s:s + _SW_WIN_B].mean()   for s in starts])
    return t, r

def _uid_boot(rate_raw):
    lam  = np.maximum(rate_raw * _BIN_S_UID * N_TRIALS_UID, 0)
    n_sw = len(range(0, len(rate_raw) - _SW_WIN_B + 1, _SW_STEP_B))
    boot = np.empty((N_BOOT_UID, n_sw))
    rng  = np.random.default_rng(42)
    for i in range(N_BOOT_UID):
        counts  = rng.poisson(lam)
        r_boot  = counts / (_BIN_S_UID * N_TRIALS_UID)
        _, sw   = _uid_slide(r_boot)
        boot[i] = sw
    return (np.percentile(boot, CI_BOUNDS_UID[0], axis=0),
            np.percentile(boot, CI_BOUNDS_UID[1], axis=0))

_blks_uid = [b for b in BLOCKER_ORDER_UID
             if (TARGET_CID, b, TARGET_PRR, TARGET_PWR) in zscore_store_all]

if not _blks_uid:
    print(f'No data for {TARGET_CID}  PRR={TARGET_PRR}  power={TARGET_PWR}')
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))

    # Stimulus rectangle (0–20 ms)
    ax.axvspan(0, 20, alpha=0.15, color='#E74C3C', linewidth=0, zorder=0)

    # Onset line
    ax.axvline(0, color='k', lw=0.8, ls='--', alpha=0.4)

    # Baseline from no-blocker condition
    nb_d = zscore_store_all.get((TARGET_CID, 'noblocker', TARGET_PRR, TARGET_PWR))
    if nb_d:
        ax.axhline(nb_d['mu'], color='#888', lw=0.8, ls=':', alpha=0.65,
                   label=f'Baseline  {nb_d["mu"]:.1f} Hz')

    # Y-offsets to avoid overlapping latency labels
    lat_y_offsets = [0.96, 0.87, 0.78, 0.69]

    for k, blk in enumerate(_blks_uid):
        d   = zscore_store_all[(TARGET_CID, blk, TARGET_PRR, TARGET_PWR)]
        col = BLOCKER_COLOR_UID[blk]

        sw_t, sw_r = _uid_slide(d['rate_raw'])
        lo, hi     = _uid_boot(d['rate_raw'])

        # Bootstrap envelope (excluded from legend)
        ax.fill_between(sw_t, lo, hi, alpha=0.20, color=col,
                        linewidth=0, label='_nolegend_')
        ax.plot(sw_t, sw_r, color=col, lw=1.8,
                label=BLOCKER_LABEL_UID[blk])

        # Latency marker — number only, same colour as trace
        lat = d.get('latency')
        if d.get('significant') and lat is not None and not np.isnan(float(lat)):
            ax.axvline(lat, color=col, lw=1.2, ls=':', alpha=0.85)
            ax.text(lat + 2, lat_y_offsets[k % len(lat_y_offsets)],
                    f'{lat:.0f} ms',
                    transform=ax.get_xaxis_transform(),
                    color=col, fontsize=7.5, va='top', ha='left', clip_on=True)

    ax.set_xlabel('Time from stimulus (ms)', fontsize=9)
    ax.set_ylabel('Firing rate (Hz)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8, framealpha=0.9, loc='upper right')

    fig.suptitle(
        f'{TARGET_CID}   ·   PRR = {int(TARGET_PRR / 1000)} kHz  ·  {_pwr_label(TARGET_PWR)}\n'
        f'20 ms sliding window / 5 ms step  ·  95 % CI  (Poisson bootstrap, {N_BOOT_UID} resamples)',
        fontsize=9.5, fontweight='bold',
    )
    plt.tight_layout()

    fname = figure_dir / f'psth_{TARGET_CID}_prr{int(TARGET_PRR)}_pwr{int(TARGET_PWR)}.png'
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fname}')
